# Dataset Preparation

Builds the project's evaluation datasets through checkpointed merging, deterministic cleaning, optional LLM rewriting, final validation, and publishing.

## Environment

In [ ]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
from IPython import get_ipython

from pprint import pformat
from typing import Any

formatter = get_ipython().display_formatter.formatters["text/plain"]
formatter.for_type(dict, lambda obj, p, _: p.text(pformat(obj, width=120, sort_dicts=False)))
formatter.for_type(list, lambda obj, p, _: p.text(pformat(obj, width=120, sort_dicts=False)))

## CounterFact

### Raw Snapshot

In [ ]:
# from datasets import load_dataset


# cfact_raw = load_dataset("azhx/counterfact")
# cfact_raw.save_to_disk("../data/datasets/counterfact/raw")

In [ ]:
# from datasets import load_from_disk


# cfact_raw = load_from_disk("../data/datasets/counterfact/raw")
# cfact_raw

### Merge And Target Validation

In [ ]:
# from datasets import concatenate_datasets


# def get_key(record: dict[str, Any]) -> tuple[str, str]:
#     rewrite = record["requested_rewrite"]

#     return (
#         rewrite["subject"],
#         rewrite["relation_id"],
#     )


# cfact_merged = concatenate_datasets([cfact_raw["train"], cfact_raw["test"]])

# target_ids = {}
# for record in cfact_merged:
#     key = get_key(record)
#     rewrite = record["requested_rewrite"]
#     ids = target_ids.setdefault(key, {"true": set(), "new": set()})
#     ids["true"].add(rewrite["target_true"]["id"])
#     ids["new"].add(rewrite["target_new"]["id"])

# valid_keys = {
#     key for key, target_ids in target_ids.items()
#     if len(target_ids["true"]) == 1 and len(target_ids["new"]) == 1
# }
# cfact_merged = cfact_merged.filter(lambda record: get_key(record) in valid_keys)

# cfact_merged.save_to_disk("../data/datasets/counterfact/merged")

In [ ]:
# from datasets import load_from_disk


# cfact_merged = load_from_disk("../data/datasets/counterfact/merged")
# cfact_merged

### Prompt Cleaning

In [ ]:
# import json
# import re
# import unicodedata
# from collections import defaultdict
#
#
# def normalize_prompt(text: str) -> str:
#     text = unicodedata.normalize("NFKC", str(text)).casefold()
#     return " ".join(text.split())
#
#
# id_token_re = re.compile(r"\b(?:ID|PID)\b")
# def is_viable_prompt(prompt: str, answer: str) -> bool:
#     if not normalize_prompt(prompt):
#         return False
#     answer_re = re.compile(rf"\b{re.escape(answer)}\b", re.IGNORECASE)
#     return not answer_re.search(prompt) and\
#            not id_token_re.search(prompt)
#
#
# def get_words(text: str) -> tuple[str, ...]:
#     text = unicodedata.normalize("NFKC", str(text)).casefold()
#     return tuple(re.findall(r"\w+", text))
#
#
# EDIT_SUBJECT_END = "__edit_subject__"
# def build_edit_subject_trie(records) -> dict[str, Any]:
#     edit_subject_trie = {}
#     for record in records:
#         rewrite = record["requested_rewrite"]
#         node = edit_subject_trie
#         for token in get_words(rewrite["subject"]):
#             node = node.setdefault(token, {})
#         node.setdefault(EDIT_SUBJECT_END, set())\
#             .add(rewrite["relation_id"])
#     return edit_subject_trie
#
#
# edit_subject_trie = build_edit_subject_trie(cfact_merged)
# def contains_edit_subject(prompt: str, relation_id: str) -> bool:
#     words = get_words(prompt)
#     for start in range(len(words)):
#         node = edit_subject_trie
#         for end in range(start, len(words)):
#             node = node.get(words[end])
#             if node is None:
#                 break
#             relations = node.get(EDIT_SUBJECT_END, set())
#             if relation_id in relations:
#                 return True
#     return False
#
#
# def build_relation_templates(records) -> dict[str, tuple[tuple[str, str], ...]]:
#     templates = defaultdict(set)
#     for record in records:
#         rewrite = record["requested_rewrite"]
#         template = rewrite["prompt"]
#         if template.count("{}") != 1:
#             raise ValueError(f"Unexpected CounterFact template: {template}")
#         templates[rewrite["relation_id"]].add(tuple(template.split("{}")))
#     return {
#         relation_id: tuple(sorted(
#             values,
#             key=lambda value: len(value[0]) + len(value[1]),
#             reverse=True,
#         ))
#         for relation_id, values in templates.items()
#     }
#
#
# relation_templates = build_relation_templates(cfact_merged)
# def extract_relation_subject(prompt: str, relation_id: str) -> str:
#     matches = []
#     for prefix, suffix in relation_templates[relation_id]:
#         if not prompt.startswith(prefix) or not prompt.endswith(suffix):
#             continue
#         end = len(prompt) - len(suffix) if suffix else len(prompt)
#         subject = prompt[len(prefix):end]
#         if subject:
#             matches.append((len(prefix) + len(suffix), subject))
#     if not matches:
#         raise ValueError(f"No CounterFact template matched: {prompt}")
#     longest = max(length for length, _ in matches)
#     subjects = {
#         normalize_prompt(subject): subject
#         for length, subject in matches
#         if length == longest
#     }
#     if len(subjects) != 1:
#         raise ValueError(f"Ambiguous CounterFact locality subject: {prompt}")
#     return next(iter(subjects.values()))
#
#
# def render_edit_prompt(record: dict[str, Any]) -> dict[str, Any]:
#     rewrite = dict(record["requested_rewrite"])
#     rewrite["prompt"] = rewrite["prompt"].format(rewrite["subject"])
#     return {"requested_rewrite": rewrite}
#
#
# def clean_record(record: dict[str, Any]) -> dict[str, Any]:
#     rewrite = record["requested_rewrite"]
#     relation_id = rewrite["relation_id"]
#     true_answer = rewrite["target_true"]["str"]
#     new_answer = rewrite["target_new"]["str"]
#
#     seen_prompts = {normalize_prompt(rewrite["prompt"])}
#
#     def clean_prompts(prompts: list[str], answer: str, locality: bool = False) -> list[str]:
#         cleaned = []
#         for prompt in prompts:
#             prompt_key = normalize_prompt(prompt)
#             if prompt_key in seen_prompts:
#                 continue
#             seen_prompts.add(prompt_key)
#             if not is_viable_prompt(prompt, answer):
#                 continue
#             if locality and contains_edit_subject(prompt, relation_id):
#                 continue
#             cleaned.append(prompt)
#         return cleaned
#
#     generation = clean_prompts(record["generation_prompts"], new_answer)
#     neighborhood = clean_prompts(record["neighborhood_prompts"], true_answer, True)
#     attribute = clean_prompts(record["attribute_prompts"], new_answer, True)
#
#     return {
#         "neighborhood_prompts": neighborhood,
#         "attribute_prompts": attribute,
#         "generation_prompts": generation,
#     }
#
#
# def find_conflicting_locality_facts(records) -> set[tuple[str, str]]:
#     answers = defaultdict(set)
#     for record in records:
#         rewrite = record["requested_rewrite"]
#         relation_id = rewrite["relation_id"]
#         for field, answer_field in (
#             ("neighborhood_prompts", "target_true"),
#             ("attribute_prompts", "target_new"),
#         ):
#             answer_id = rewrite[answer_field]["id"]
#             for prompt in record[field]:
#                 subject = extract_relation_subject(prompt, relation_id)
#                 answers[(normalize_prompt(subject), relation_id)].add(answer_id)
#     return {
#         key for key, answer_ids in answers.items()
#         if len(answer_ids) > 1
#     }
#
#
# def remove_conflicting_locality(record: dict[str, Any]) -> dict[str, Any]:
#     relation_id = record["requested_rewrite"]["relation_id"]
#
#     def keep(prompt: str) -> bool:
#         subject = extract_relation_subject(prompt, relation_id)
#         return (normalize_prompt(subject), relation_id) not in conflicting_locality_facts
#
#     return {
#         "neighborhood_prompts": [
#             prompt for prompt in record["neighborhood_prompts"] if keep(prompt)
#         ],
#         "attribute_prompts": [
#             prompt for prompt in record["attribute_prompts"] if keep(prompt)
#         ],
#     }
#
#
# def add_locality_subjects(record: dict[str, Any]) -> dict[str, Any]:
#     relation_id = record["requested_rewrite"]["relation_id"]
#     return {
#         "neighborhood_subjects": [
#             extract_relation_subject(prompt, relation_id)
#             for prompt in record["neighborhood_prompts"]
#         ],
#         "attribute_subjects": [
#             extract_relation_subject(prompt, relation_id)
#             for prompt in record["attribute_prompts"]
#         ],
#     }
#
#
# cfact_clean = cfact_merged\
#     .remove_columns(["case_id", "pararel_idx", "paraphrase_prompts"])\
#     .map(render_edit_prompt)\
#     .filter(lambda record: is_viable_prompt(
#         record["requested_rewrite"]["prompt"],
#         record["requested_rewrite"]["target_new"]["str"]
#     ))\
#     .map(clean_record)
#
# locality_before = sum(
#     len(record["neighborhood_prompts"]) + len(record["attribute_prompts"])
#     for record in cfact_clean
# )
# conflicting_locality_facts = find_conflicting_locality_facts(cfact_clean)
# cfact_clean = cfact_clean\
#     .map(remove_conflicting_locality)\
#     .map(add_locality_subjects)
# locality_after = sum(
#     len(record["neighborhood_prompts"]) + len(record["attribute_prompts"])
#     for record in cfact_clean
# )
# clean_report = {
#     "rows": len(cfact_clean),
#     "conflicting_locality_facts": len(conflicting_locality_facts),
#     "locality_before": locality_before,
#     "locality_removed": locality_before - locality_after,
#     "locality_after": locality_after,
# }
#
# clean_path = "../data/datasets/counterfact/clean"
# cfact_clean.save_to_disk(clean_path)
# with open(f"{clean_path}/report.json", "w", encoding="utf-8") as file:
#     json.dump(clean_report, file, ensure_ascii=False, indent=2)
# clean_report

In [ ]:
# from datasets import load_from_disk


# cfact_clean = load_from_disk("../data/datasets/counterfact/clean")
# cfact_clean

### Evaluation Sampling

In [ ]:
# import unicodedata
# from random import Random


# SEED = 42
# N_GENERATION = 3
# N_NEIGHBORHOOD = 2
# N_ATTRIBUTE = 2


# def sample_prompts(prompts: list[str], n: int, seed_key: str) -> list[str]:
#     prompts = list(prompts)
#     Random(f"{SEED}:{seed_key}").shuffle(prompts)
#     return prompts[:n]
#
#
# def locality_subject_key(subject: str) -> str:
#     subject = unicodedata.normalize("NFKC", str(subject)).casefold()
#     return " ".join(subject.split())
#
#
# def count_locality_facts(subjects: list[str]) -> int:
#     return len({locality_subject_key(subject) for subject in subjects})
#
#
# def sample_locality(
#     prompts: list[str],
#     subjects: list[str],
#     n: int,
#     seed_key: str,
# ) -> list[tuple[str, str]]:
#     candidates = list(zip(prompts, subjects))
#     Random(f"{SEED}:{seed_key}").shuffle(candidates)
#
#     selected = []
#     seen_subjects = set()
#     for prompt, subject in candidates:
#         subject_key = locality_subject_key(subject)
#         if subject_key in seen_subjects:
#             continue
#         seen_subjects.add(subject_key)
#         selected.append((prompt, subject))
#         if len(selected) == n:
#             break
#     return selected
#
#
# def has_uniform_candidates(record: dict[str, Any]) -> bool:
#     return (
#         len(record["generation_prompts"]) >= N_GENERATION and
#         count_locality_facts(record["neighborhood_subjects"]) >= N_NEIGHBORHOOD and
#         count_locality_facts(record["attribute_subjects"]) >= N_ATTRIBUTE
#     )


# def uniform(record: dict[str, Any], idx: int) -> dict[str, Any]:
#     rewrite = record["requested_rewrite"]

#     true_answer = rewrite["target_true"]
#     new_answer = rewrite["target_new"]

#     generation = sample_prompts(
#         record["generation_prompts"],
#         N_GENERATION,
#         f"{idx}:generation",
#     )

#     neighborhood = sample_locality(
#         record["neighborhood_prompts"],
#         record["neighborhood_subjects"],
#         N_NEIGHBORHOOD,
#         f"{idx}:neighborhood",
#     )

#     attribute = sample_locality(
#         record["attribute_prompts"],
#         record["attribute_subjects"],
#         N_ATTRIBUTE,
#         f"{idx}:attribute",
#     )

#     return {
#         "edit": {
#             "prompt": rewrite["prompt"],
#             "relation_id": rewrite["relation_id"],
#             "subject": rewrite["subject"],
#             "answer": {
#                 "true": true_answer,
#                 "new": new_answer,
#             },
#         },
#         "generality": generation,
#         "locality": [
#             {
#                 "prompt": prompt,
#                 "subject": subject,
#                 "answer": true_answer,
#             }
#             for prompt, subject in neighborhood
#         ] + [
#             {
#                 "prompt": prompt,
#                 "subject": subject,
#                 "answer": new_answer,
#             }
#             for prompt, subject in attribute
#         ],
#     }


# cfact_uniform = cfact_clean\
#     .filter(has_uniform_candidates)\
#     .map(
#         uniform,
#         with_indices=True,
#         remove_columns=cfact_clean.column_names
#     )

# cfact_uniform.save_to_disk("../data/datasets/counterfact/uniform")

In [ ]:
# from datasets import load_from_disk
#
#
# cfact_uniform = load_from_disk("../data/datasets/counterfact/uniform")
# cfact_uniform

### Prompt Enhancement

In [ ]:
# import asyncio
# import json
# import re
# import unicodedata
#
# from datasets import Dataset
# from openai import AsyncOpenAI
# from typing import Any
# from tqdm.auto import tqdm
#
#
# def normalize_prompt(text: str) -> str:
#     text = unicodedata.normalize("NFKC", str(text))
#     return " ".join(text.split())
#
#
# def prompt_key(text: str) -> str:
#     return normalize_prompt(text).casefold()
#
#
# def contains_exact_text(prompt: str, text: str, ignore_case: bool = False) -> bool:
#     prompt = normalize_prompt(prompt)
#     text = normalize_prompt(text)
#     if not text:
#         return False
#     flags = re.IGNORECASE if ignore_case else 0
#     return re.search(rf"(?<!\w){re.escape(text)}(?!\w)", prompt, flags) is not None
#
#
# def answer_leaks_into_prompt(prompt: str, answer: str) -> bool:
#     return contains_exact_text(prompt, answer, ignore_case=True)
#
#
# def strip_json_response(text: str) -> str:
#     text = text.strip().lstrip("\ufeff")
#     fenced = re.fullmatch(r"```(?:json)?\s*(.*?)\s*```", text, re.IGNORECASE | re.DOTALL)
#     return fenced.group(1).strip() if fenced else text
#
#
# def to_enhancement_record(record: dict[str, Any]) -> dict[str, Any]:
#     return {
#         "edit": {
#             "prompt": record["edit"]["prompt"],
#             "subject": record["edit"]["subject"],
#             "answer": {
#                 "true": record["edit"]["answer"]["true"]["str"],
#                 "new": record["edit"]["answer"]["new"]["str"],
#             },
#         },
#         "generality": list(record["generality"]),
#         "locality_subjects": [
#             item["subject"] for item in record["locality"]
#         ],
#         "locality": [
#             {"prompt": item["prompt"], "answer": item["answer"]["str"]}
#             for item in record["locality"]
#         ],
#     }
#
#
# def restore_enhanced_record(source: dict[str, Any], enhanced: dict[str, Any]) -> dict[str, Any]:
#     return {
#         "edit": {
#             "prompt": normalize_prompt(enhanced["edit"]["prompt"]),
#             "relation_id": source["edit"]["relation_id"],
#             "subject": source["edit"]["subject"],
#             "answer": source["edit"]["answer"],
#         },
#         "generality": [
#             normalize_prompt(prompt) for prompt in enhanced["generality"]
#         ],
#         "locality": [
#             {
#                 "prompt": normalize_prompt(output["prompt"]),
#                 "subject": item["subject"],
#                 "answer": item["answer"],
#             }
#             for item, output in zip(source["locality"], enhanced["locality"])
#         ],
#     }
#
#
# def get_enhancement_examples() -> list[dict[str, Any]]:
#     return [
#         {
#             "purpose": "Repair the entire record. Every weak edit, generality, and locality prompt is improved while preserving the mother-tongue relation.",
#             "input": {
#                 "edit": {
#                     "prompt": "Clara Voss spoke the language",
#                     "subject": "Clara Voss",
#                     "answer": {"true": "German", "new": "Spanish"},
#                 },
#                 "generality": [
#                     "Where Clara Voss was born, people speak the language of",
#                     "Clara Voss was born in",
#                     "Clara Voss, a native",
#                 ],
#                 "locality_subjects": ["Martin Keller", "Elise Bauer", "Rafael Moreno", "Lucia Torres"],
#                 "locality": [
#                     {"prompt": "Martin Keller, speaker of", "answer": "German"},
#                     {"prompt": "Elise Bauer spoke the language", "answer": "German"},
#                     {"prompt": "Rafael Moreno, a native", "answer": "Spanish"},
#                     {"prompt": "Lucia Torres spoke the language", "answer": "Spanish"},
#                 ],
#             },
#             "output": {
#                 "edit": {
#                     "prompt": "Clara Voss's mother tongue is",
#                     "subject": "Clara Voss",
#                     "answer": {"true": "German", "new": "Spanish"},
#                 },
#                 "generality": [
#                     "Clara Voss's native language is",
#                     "The first language Clara Voss learned was",
#                     "The language Clara Voss first spoke was",
#                 ],
#                 "locality_subjects": ["Martin Keller", "Elise Bauer", "Rafael Moreno", "Lucia Torres"],
#                 "locality": [
#                     {"prompt": "Martin Keller's native language is", "answer": "German"},
#                     {"prompt": "The mother tongue of Elise Bauer is", "answer": "German"},
#                     {"prompt": "Rafael Moreno's native language is", "answer": "Spanish"},
#                     {"prompt": "Lucia Torres's mother tongue is", "answer": "Spanish"},
#                 ],
#             },
#         },
#         {
#             "purpose": "Repair under-directed fragments across edit, generality, and locality while preserving cloze type and instrument relation.",
#             "input": {
#                 "edit": {
#                     "prompt": "Nolan Reed, the",
#                     "subject": "Nolan Reed",
#                     "answer": {"true": "guitar", "new": "piano"},
#                 },
#                 "generality": [
#                     "Nolan Reed plays",
#                     "Nolan Reed is known for",
#                     "Nolan Reed's sound is based around the",
#                 ],
#                 "locality_subjects": ["Ari Lane", "Sofia Vale", "Mara Chen", "Kai Soto"],
#                 "locality": [
#                     {"prompt": "Ari Lane, the", "answer": "guitar"},
#                     {"prompt": "Sofia Vale's music uses the", "answer": "guitar"},
#                     {"prompt": "Mara Chen, the", "answer": "piano"},
#                     {"prompt": "Kai Soto performs", "answer": "piano"},
#                 ],
#             },
#             "output": {
#                 "edit": {
#                     "prompt": "Nolan Reed's primary instrument is the",
#                     "subject": "Nolan Reed",
#                     "answer": {"true": "guitar", "new": "piano"},
#                 },
#                 "generality": [
#                     "The instrument Nolan Reed plays is the",
#                     "Nolan Reed is known for playing the",
#                     "Nolan Reed's main instrument is the",
#                 ],
#                 "locality_subjects": ["Ari Lane", "Sofia Vale", "Mara Chen", "Kai Soto"],
#                 "locality": [
#                     {"prompt": "Ari Lane's primary instrument is the", "answer": "guitar"},
#                     {"prompt": "Sofia Vale's main instrument is the", "answer": "guitar"},
#                     {"prompt": "Mara Chen's primary instrument is the", "answer": "piano"},
#                     {"prompt": "The instrument Kai Soto performs on is the", "answer": "piano"},
#                 ],
#             },
#         },
#         {
#             "purpose": "Repair all prompts while keeping citizenship as legal citizenship, not residence, birthplace, passport ownership, ethnicity, or workplace.",
#             "input": {
#                 "edit": {
#                     "prompt": "Mira Solberg is from",
#                     "subject": "Mira Solberg",
#                     "answer": {"true": "Norway", "new": "Belgium"},
#                 },
#                 "generality": [
#                     "Mira Solberg currently lives in",
#                     "The passport that Mira Solberg carries is from",
#                     "Mira Solberg belongs to the country of",
#                 ],
#                 "locality_subjects": ["Jonas Eide", "Ingrid Dahl", "Elise Laurent", "René Martel"],
#                 "locality": [
#                     {"prompt": "Jonas Eide is from", "answer": "Norway"},
#                     {"prompt": "Ingrid Dahl belongs to the country of", "answer": "Norway"},
#                     {"prompt": "Elise Laurent lives in", "answer": "Belgium"},
#                     {"prompt": "René Martel has papers from", "answer": "Belgium"},
#                 ],
#             },
#             "output": {
#                 "edit": {
#                     "prompt": "Mira Solberg holds citizenship from",
#                     "subject": "Mira Solberg",
#                     "answer": {"true": "Norway", "new": "Belgium"},
#                 },
#                 "generality": [
#                     "Mira Solberg is a citizen of",
#                     "The country of citizenship of Mira Solberg is",
#                     "Mira Solberg legally holds citizenship from",
#                 ],
#                 "locality_subjects": ["Jonas Eide", "Ingrid Dahl", "Elise Laurent", "René Martel"],
#                 "locality": [
#                     {"prompt": "Jonas Eide holds citizenship from", "answer": "Norway"},
#                     {"prompt": "The country of citizenship of Ingrid Dahl is", "answer": "Norway"},
#                     {"prompt": "Elise Laurent is a citizen of", "answer": "Belgium"},
#                     {"prompt": "The citizenship held by René Martel is from", "answer": "Belgium"},
#                 ],
#             },
#         },
#         {
#             "purpose": "Fail instead of cosmetically rewriting duplicate locality facts. The first two locality prompts ask the same subject-relation-answer fact.",
#             "input": {
#                 "edit": {
#                     "prompt": "The instrument Nolan Reed plays is the",
#                     "subject": "Nolan Reed",
#                     "answer": {"true": "guitar", "new": "piano"},
#                 },
#                 "generality": [
#                     "Nolan Reed is known for playing the",
#                     "Nolan Reed performs on the",
#                     "Nolan Reed's main instrument is the",
#                 ],
#                 "locality_subjects": ["Ari Lane", "Ari Lane", "Mara Chen", "Kai Soto"],
#                 "locality": [
#                     {"prompt": "Ari Lane plays the", "answer": "guitar"},
#                     {"prompt": "Ari Lane performs on the", "answer": "guitar"},
#                     {"prompt": "Mara Chen plays the", "answer": "piano"},
#                     {"prompt": "The instrument Kai Soto performs on is the", "answer": "piano"},
#                 ],
#             },
#             "output": {"status": "failed", "reason": "locality contains repeated subject-relation-answer facts"},
#         },
#     ]
#
#
# ENHANCEMENT_EXAMPLES = json.dumps(
#     get_enhancement_examples(), ensure_ascii=False, indent=2
# )
#
#
# RELATION_GUIDANCE = {
#     "P19": "place of birth; every prompt must explicitly ask where the subject was born or for its birthplace, never origin, residence, parents, nationality, or regional language",
#     "P37": "an officially recognized language; do not reduce it to a language merely spoken, understood, or commonly used, and do not imply uniqueness",
#     "P101": "field, domain, or focus of work; use answer-compatible expertise, activity, scholarly, professional, or creative-focus wording when literal field wording is unnatural, without drifting to employer, work location, or formal position",
#     "P103": "native language, mother tongue, or first language; never infer it from birthplace, nationality, origin, or any language the subject merely speaks or writes",
#     "P106": "occupation or profession; use article-free wording because the prompt must fit both answers, and do not change it into field of work, employer, position held, accomplishment, or known-for wording",
#     "P1412": "a language the subject speaks, writes, signs, or otherwise uses; prefer modality-neutral wording unless the modality is supported, and do not change it into native language, birthplace, residence, nationality, or friends' language",
#     "P176": "manufacturer; ask directly who manufactured or produced the subject, not who created, developed, sold, owned, or supervised production of it",
#     "P178": "developer; ask directly who developed the subject, not its manufacturer, owner, distributor, seller, or generic creator",
#     "P276": "physical location; ask directly where the subject is located, preserving available geographic granularity and avoiding navigation, surroundings, restaurants, headquarters, country, or administrative-containment drift",
#     "P131": "administrative territorial entity containing the subject; explicitly ask which administrative territory contains it, never nearby places, restaurants, surroundings, navigation, origin, or headquarters",
#     "P495": "country of origin; ask directly for the country of origin and avoid medium-specific created, developed, or formulated wording unless natural for the subject; never ask regional language, formation location, publication place, nationality, or headquarters",
#     "P740": "formation or founding location; use formed, founded, or established for organizations and groups, but use the closest answer-compatible origin wording for subjects that cannot naturally be founded; never invent founders or inspiration",
#     "P937": "place where the subject works or is professionally based; ask directly for work location, not employer, residence, birthplace, commute, nearby surroundings, office meals, or lunch",
#     "P1303": "musical instrument, voice, or performance technique used by the subject; choose answer-compatible plays, sings, uses, or neutral performance-medium wording, never genre, occupation, vague known-for/skill wording, subjective praise, or incomplete apposition",
#     "P17": "country containing or associated with the subject; explicitly ask which country, never a city, region, birthplace, origin, navigation route, surroundings, or restaurants",
#     "P20": "place of death; ask directly where the subject died or passed away, without inventing that they were killed or changing it into burial place, final residence, local events, or regional language",
#     "P27": "country of citizenship; prefer citizen of or holds citizenship in, without implying uniqueness or changing it into birthplace, residence, ethnicity, origin, passport ownership, or native language",
#     "P30": "continent; explicitly ask which continent contains or is associated with the subject, never country, region, navigation, surroundings, or regional language",
#     "P36": "capital city; ask directly for the subject's capital, never headquarters, administrative office, tourist attractions, reputation, or language spoken there",
#     "P39": "formal position, office, title, or appointed role held; prefer holds the office/title of or serves as, and do not reduce it to a general field, accomplishment, or playing position",
#     "P108": "employer or employing organization; use employed by, works for, or professionally affiliated with, never a bare of fragment, work location, occupation, or unsupported narrative about work",
#     "P127": "owner; explicitly ask who or what owns the subject, never vague by/from wording, seller, trademark owner, manufacturer, developer, creator, or member organization",
#     "P136": "genre or the closest medium-specific classification; choose wording that fits the subject and answer, such as musical style or type of music, literary genre or type of book, screen genre, theatrical form, or a neutral type/style/category when the medium is unclear. Do not force the literal word genre or reuse templates based on play, perform, greatest artistic work, or incredible at. Preserve the classification meaning as closely as possible without drifting to instrument, occupation, creator, or a specific work",
#     "P138": "name source or namesake; ask what person, place, entity, material, or concept the subject was named after, without assuming commemoration or using known-for and incomplete reason clauses",
#     "P140": "religion, faith, religious identity, or tradition associated with the subject; use neutral wording that fits people and institutions and noun or adjective answers, without introducing official status, worship, attendance, values, ethnicity, or nationality",
#     "P159": "headquarters location; ask directly where the subject is headquartered and use the plural noun headquarters, never navigation, surroundings, restaurants, general location, origin, or formation location",
#     "P190": "twinned administrative counterpart; ask directly which city, town, region, or administrative body the subject is twinned with, never the counterpart's language, attractions, reputation, or another attribute",
#     "P264": "record label associated with the subject; distinguish a performer signed to or represented by a label from a work released on or by a label, using neutral associated-label wording when uncertain; never ownership, sales, contracts, broadcaster, or manufacturer",
#     "P364": "original language of an audiovisual work; preserve the original qualifier and use originally made or produced in when natural, never treat the title as a place or ask production country, filming location, inhabitants' language, or creator language",
#     "P407": "language used by the work or name; use written in only for clearly textual subjects and otherwise prefer neutral language used in/by, without inferring author origin, country, creator language, or originality",
#     "P413": "sport playing position or on-field role; ask directly which position the athlete plays using sport-neutral wording, never expertise, strength, occupation, formal position held, activity, or incomplete apposition",
#     "P449": "original network, channel, broadcaster, or distribution platform on which the subject first aired or premiered; make that role explicit and never ask for another program, add subjective opinions, or change it into production company or release location",
#     "P463": "membership in a group, organization, alliance, federation, team, or band; prefer direct member-of wording and never weaken it into generic affiliation, ownership, employment, mereological part-of, or known-for wording",
# }
#
#
# def build_enhancement_prompt(record: dict[str, Any], relation_id: str) -> str:
#     payload = json.dumps(record, ensure_ascii=False, indent=2)
#     relation_guidance = RELATION_GUIDANCE.get(
#         relation_id,
#         f"Wikidata relation {relation_id}; preserve its exact meaning when natural, otherwise use the closest answer-compatible paraphrase",
#     )
#     return (
#         "Repair one CounterFact evaluation record.\n"
#         "Only edit prompt strings. All answers, subjects, list lengths, and list order are locked.\n"
#         f"Locked relation for every prompt in this record: {relation_guidance}.\n"
#         "The relation identifier is context only. Do not add it to the output.\n"
#         "You must inspect and repair the whole record: edit.prompt, every generality prompt, and every locality prompt.\n"
#         "Do not assume only generality needs rewriting. If a locality or edit prompt is noisy, under-directed, or relation-drifted, repair it too.\n"
#         "Leave prompts unchanged when they are already clear and correct; otherwise make the smallest safe repair.\n"
#         "The input generality strings come from CounterFact generation prompts and are not trusted paraphrases. Treat their wording as raw material, not semantics that must be preserved.\n"
#         "Rewrite every generality string into a genuine paraphrase of edit.prompt: it must query the exact same subject-relation fact, keep the exact edit subject, and make edit.answer.new the direct answer.\n"
#         "Generality prompts must differ meaningfully in wording, but must not test residence, origin, nearby places, skills, ownership, or any other neighboring fact unless that is the locked edit relation.\n"
#         "locality_subjects gives the exact subject of each locality item in matching order. Return it unchanged, and keep each subject verbatim in its prompt.\n"
#         "Locality is a list of prompt-answer objects. Every locality prompt uses the same locked relation but keeps its own subject and is judged only against its own assigned answer.\n"
#         "For a successful repair, return one JSON object with exactly the same schema as the input. "
#         "If the record cannot be repaired without answer leakage, relation drift, or semantic duplicate locality facts, "
#         "return exactly {\"status\": \"failed\", \"reason\": \"short reason\"}.\n\n"
#         "Central repair goal:\n"
#         "Each prompt must make its assigned answer the direct, natural, preferred answer. "
#         "Repair noisy fragments and relation drift. Prefer the exact intended fact, but when literal relation wording "
#         "cannot form a natural prompt for the subject-answer pair, use the closest logically equivalent wording. "
#         "The locked record relation defines the intended fact type for edit, generality, and locality. Locality differs only in subject and assigned answer, not relation.\n\n"
#         "Relation discipline:\n"
#         "- Distinguish the intended relation from nearby concepts such as birthplace versus nationality, creator versus owner, or instrument versus occupation.\n"
#         "- Preserve exact relation meaning when it can be expressed naturally. If it cannot, use the nearest answer-compatible synonym, category, or medium-specific wording.\n"
#         "- For generality, the locked edit relation always overrides the input prompt's apparent relation. Replace conflicting predicates instead of preserving their meaning.\n"
#         "- If an edit or locality prompt drifts to a nearby relation, repair it toward the intended meaning without changing its entities or answer.\n"
#         "- Apply the locked relation to locality as well. Repair ambiguous or drifting locality wording instead of trusting its apparent predicate.\n\n"
#         "Rules:\n"
#         "1) Do not change any answer string, including edit.answer.true, edit.answer.new, or locality[*].answer.\n"
#         "2) Do not change edit.subject. Keep the exact full subject with the same casing in edit.prompt and every generality prompt.\n"
#         "3) Do not add, remove, replace, or merge named entities in any prompt.\n"
#         "4) You may change between cloze and question form when that produces a clearer, more natural prompt.\n"
#         "5) Keep cloze prompts as natural prefixes ending immediately before the answer. Do not use blanks or placeholders.\n"
#         "6) Keep question prompts as direct questions ending with a question mark.\n"
#         "7) Do not include meta-instructions such as 'answer with', 'respond with', 'in one word', or output-format instructions.\n"
#         "8) The assigned evaluation answer must not appear verbatim inside its prompt. For edit.prompt and generality prompts, only edit.answer.new is forbidden because those prompts are judged after editing. For locality prompts, locality[*].answer is forbidden.\n"
#         "9) Avoid duplicate or near-duplicate wording within the whole record, including edit, generality, and locality. "
#         "Generality prompts intentionally test the edited fact, but their wording must still be meaningfully different.\n"
#         "10) Within locality, if two prompts query the same subject-relation-answer fact, return failed instead of saving a cosmetic paraphrase.\n"
#         "11) Prefer concise natural wording over noisy sentence fragments.\n"
#         "12) Reject the record when an answer is questionable, wrong-type, ambiguous, or cannot be made the direct answer without changing the underlying fact.\n\n"
#         "Synthetic full-record examples. Follow the behavior and schema, not the names or facts. "
#         "Notice that successful examples repair edit, generality, and locality prompts when they are weak.\n"
#         f"{ENHANCEMENT_EXAMPLES}\n\n"
#         "Input JSON:\n"
#         f"{payload}\n\n"
#         "Output JSON only:"
#     )
#
#
# def validate_enhanced_record(source: dict[str, Any], enhanced) -> list[str]:
#     errors: list[str] = []
#
#     if not isinstance(enhanced, dict):
#         return ["output is not a JSON object"]
#
#     if set(enhanced) != {"edit", "generality", "locality_subjects", "locality"}:
#         return ["output keys must be exactly: edit, generality, locality_subjects, locality"]
#     if not isinstance(enhanced["edit"], dict):
#         return ["edit is not an object"]
#     if set(enhanced["edit"]) != {"prompt", "subject", "answer"}:
#         return ["edit keys must be exactly: prompt, subject, answer"]
#     if not isinstance(enhanced["generality"], list):
#         return ["generality is not a list"]
#     if not isinstance(enhanced["locality_subjects"], list):
#         return ["locality_subjects is not a list"]
#     if not isinstance(enhanced["locality"], list):
#         return ["locality is not a list"]
#
#     if enhanced["edit"].get("subject") != source["edit"].get("subject"):
#         errors.append("edit.subject changed")
#     if enhanced["edit"].get("answer") != source["edit"].get("answer"):
#         errors.append("edit.answer changed")
#     if not isinstance(enhanced["edit"].get("prompt"), str):
#         errors.append("edit.prompt is not a string")
#
#     subject = source["edit"].get("subject", "")
#     edit_prompt = enhanced["edit"].get("prompt", "")
#     if subject and not contains_exact_text(edit_prompt, subject):
#         errors.append("edit.prompt missing exact subject")
#
#     if len(enhanced.get("generality", [])) != len(source.get("generality", [])):
#         errors.append("generality length changed")
#     else:
#         for idx, out_prompt in enumerate(enhanced["generality"]):
#             if not isinstance(out_prompt, str):
#                 errors.append(f"generality[{idx}] is not a prompt string")
#             elif subject and not contains_exact_text(out_prompt, subject):
#                 errors.append(f"generality[{idx}] missing exact subject")
#
#     locality_subjects = source.get("locality_subjects", [])
#     if enhanced.get("locality_subjects") != locality_subjects:
#         errors.append("locality_subjects changed")
#     if len(locality_subjects) != len(source.get("locality", [])):
#         errors.append("locality_subjects length does not match locality")
#
#     if len(enhanced.get("locality", [])) != len(source.get("locality", [])):
#         errors.append("locality length changed")
#     else:
#         for idx, (src_item, out_item) in enumerate(zip(source["locality"], enhanced["locality"])):
#             if not isinstance(out_item, dict):
#                 errors.append(f"locality[{idx}] is not a prompt-answer object")
#             elif set(out_item) != {"prompt", "answer"}:
#                 errors.append(f"locality[{idx}] keys must be exactly: prompt, answer")
#             elif not isinstance(out_item.get("prompt"), str):
#                 errors.append(f"locality[{idx}].prompt is not a string")
#             elif out_item.get("answer") != src_item.get("answer"):
#                 errors.append(f"locality[{idx}].answer changed")
#             elif idx < len(locality_subjects) and not contains_exact_text(
#                 out_item["prompt"], locality_subjects[idx]
#             ):
#                 errors.append(f"locality[{idx}].prompt missing exact subject")
#
#     if errors:
#         return errors
#
#     prompt_pairs = [("edit", enhanced["edit"].get("prompt", ""), [source["edit"]["answer"]["new"]])]
#     prompt_pairs.extend(
#         (f"generality[{idx}]", out, [source["edit"]["answer"]["new"]])
#         for idx, out in enumerate(enhanced["generality"])
#         if isinstance(out, str)
#     )
#     prompt_pairs.extend(
#         (f"locality[{idx}]", out["prompt"], [src["answer"]])
#         for idx, (src, out) in enumerate(zip(source["locality"], enhanced["locality"]))
#         if isinstance(src, dict) and isinstance(out, dict)
#     )
#
#     seen: dict[str, str] = {}
#     for label, out_prompt, forbidden_answers in prompt_pairs:
#         prompt = normalize_prompt(out_prompt)
#         if not prompt:
#             errors.append(f"{label}.prompt is empty")
#             continue
#
#         lowered = prompt.casefold()
#         if any(token in lowered for token in (
#             "{}", "____", "[blank]", "<blank>", "[mask]",
#             "answer with", "respond with", "in one word",
#         )):
#             errors.append(f"{label}.prompt contains forbidden formatting/meta text")
#         if re.search(r"\b(?:ID|PID)\b", prompt):
#             errors.append(f"{label}.prompt contains ID/PID")
#
#         leaked_answers = [answer for answer in forbidden_answers if answer_leaks_into_prompt(prompt, answer)]
#         if leaked_answers:
#             errors.append(f"{label}.prompt leaks answer(s): {leaked_answers}")
#
#         key = prompt_key(prompt)
#         if key in seen:
#             errors.append(f"duplicate prompt: {seen[key]} and {label}")
#         else:
#             seen[key] = label
#
#     return errors
#
#
# async def enhance_record(
#     record: dict[str, Any],
#     client: AsyncOpenAI,
#     config: dict[str, Any],
#     feedback: list[str] | None = None,
# ) -> tuple[dict[str, Any] | None, list[str], Any]:
#     request_record = to_enhancement_record(record)
#     messages = [
#         {
#             "role": "system",
#             "content": "You repair benchmark prompts. You only change prompt text and never change answers.",
#         },
#         {
#             "role": "user",
#             "content": build_enhancement_prompt(
#                 request_record, record["edit"]["relation_id"]
#             ) + (
#                 "\n\nPrevious attempt failed validation. Fix every issue below and return only corrected JSON:\n"
#                 + "\n".join(f"- {error}" for error in feedback)
#                 if feedback else ""
#             ),
#         },
#     ]
#
#     completion = await client.chat.completions.create(
#         model=config["model"],
#         messages=messages,
#         **config.get("request_config", {}),
#     )
#
#     content = completion.choices[0].message.content
#     if not isinstance(content, str) or not content.strip():
#         return None, ["empty model response"], None
#
#     response_text = strip_json_response(content)
#     try:
#         enhanced = json.loads(response_text)
#     except Exception as exc:
#         return None, [f"invalid JSON: {exc}"], {"raw_response": response_text}
#
#     if isinstance(enhanced, dict) and enhanced.get("status") == "failed":
#         reason = enhanced.get("reason", "no reason")
#         return None, [f"model returned failed status: {reason}"], enhanced
#
#     errors = validate_enhanced_record(request_record, enhanced)
#     if errors:
#         return None, errors, enhanced
#
#     return restore_enhanced_record(record, enhanced), [], None
#
#
# async def enhance_dataset(
#     dataset,
#     config: dict[str, Any],
#     start: int = 0,
#     limit: int | None = None,
# ) -> tuple[Dataset, list[dict[str, Any]]]:
#     if start < 0:
#         raise ValueError("start must be non-negative")
#     if limit is not None and limit < 0:
#         raise ValueError("limit must be non-negative or None")
#
#     end = len(dataset) if limit is None else min(len(dataset), start + limit)
#     indices = list(range(start, end))
#
#     max_concurrency = config.get("max_concurrency", 4)
#     max_retries = config.get("max_retries", 3)
#     retry_delay = config.get("retry_delay", 2.0)
#     if max_concurrency <= 0:
#         raise ValueError("max_concurrency must be positive")
#     if max_retries < 0 or retry_delay < 0:
#         raise ValueError("max_retries and retry_delay must be non-negative")
#     if not indices:
#         return dataset.select([]), []
#
#     client = AsyncOpenAI(
#         base_url=config.get("base_url"),
#         api_key=config.get("api_key"),
#     )
#
#     enhanced_rows: list[dict | None] = [None] * len(indices)
#     failures: list[dict] = []
#     queue = asyncio.Queue()
#     for slot, row_idx in enumerate(indices):
#         queue.put_nowait((slot, row_idx))
#     progress = tqdm(total=len(indices), desc="Enhancing CounterFact")
#
#     async def worker() -> None:
#         while True:
#             try:
#                 slot, row_idx = queue.get_nowait()
#             except asyncio.QueueEmpty:
#                 return
#
#             record = dataset[row_idx]
#             last_errors: list[str] = []
#             last_rejected_output: object | None = None
#             feedback: list[str] | None = None
#             try:
#                 for attempt in range(max_retries + 1):
#                     try:
#                         enhanced, errors, rejected_output = await enhance_record(
#                             record, client, config, feedback
#                         )
#                     except Exception as exc:
#                         enhanced, errors, rejected_output = None, [repr(exc)], None
#
#                     if enhanced is not None:
#                         enhanced_rows[slot] = enhanced
#                         break
#
#                     last_errors = errors
#                     last_rejected_output = rejected_output
#                     feedback = errors
#                     if errors and all(
#                         error.startswith("model returned failed status:")
#                         for error in errors
#                     ):
#                         break
#                     if attempt < max_retries:
#                         await asyncio.sleep(retry_delay * (attempt + 1))
#
#                 if enhanced_rows[slot] is None:
#                     failures.append({
#                         "index": row_idx,
#                         "errors": last_errors,
#                         "record": to_enhancement_record(record),
#                         "enhanced": last_rejected_output,
#                     })
#             finally:
#                 queue.task_done()
#                 progress.update(1)
#
#     tasks = [
#         asyncio.create_task(worker())
#         for _ in range(min(max_concurrency, len(indices)))
#     ]
#     try:
#         await asyncio.gather(*tasks)
#     finally:
#         for task in tasks:
#             if not task.done():
#                 task.cancel()
#         await asyncio.gather(*tasks, return_exceptions=True)
#         progress.close()
#         await client.close()
#
#     kept_rows = [row for row in enhanced_rows if row is not None]
#     enhanced_dataset = (
#         Dataset.from_list(kept_rows, features=dataset.features)
#         if kept_rows else dataset.select([])
#     )
#     failures.sort(key=lambda item: item["index"])
#
#     return enhanced_dataset, failures

In [ ]:
# import os
# from typing import Any
# from dotenv import load_dotenv
#
#
# load_dotenv()
#
# enhancement_config = {
#     "base_url": "http://localhost:20128/v1",
#     "api_key": os.environ["9ROUTER_API_KEY"],
#     "model": "cx/gpt-5.6-luna(low)",
#     "max_concurrency": 50,
#     "max_retries": 5,
#     "retry_delay": 2.0,
# }
#
# cfact_enhanced, enhancement_failures = await enhance_dataset(
#     cfact_uniform,
#     enhancement_config,
#     start=0,
#     limit=None,
# )

In [ ]:
# import json
#
#
# enhanced_path = "../data/datasets/counterfact/enhanced"
# cfact_enhanced.save_to_disk(enhanced_path)
#
# with open(f"{enhanced_path}/failures.json", "w", encoding="utf-8") as file:
#     json.dump(enhancement_failures, file, ensure_ascii=False, indent=2)

In [ ]:
# from datasets import load_from_disk
#
#
# cfact_enhanced = load_from_disk("../data/datasets/counterfact/enhanced")
# cfact_enhanced

### Global Finalization

In [ ]:
# import asyncio
# import json
# import re
# from collections import defaultdict
#
# from datasets import Dataset
# from openai import AsyncOpenAI
# from tqdm.auto import tqdm
#
#
# EDIT_SUBJECT_END = "__edit_subject_end__"
#
#
# def get_words(text: str) -> tuple[str, ...]:
#     return tuple(re.findall(r"\w+", prompt_key(text)))
#
#
# def build_edit_subject_trie(records) -> dict[str, Any]:
#     edit_subject_trie = {}
#     for record_index, record in enumerate(records):
#         edit = record["edit"]
#         node = edit_subject_trie
#         for word in get_words(edit["subject"]):
#             node = node.setdefault(word, {})
#         node.setdefault(EDIT_SUBJECT_END, []).append({
#             "record_index": record_index,
#             "edit_subject": edit["subject"],
#             "edit_subject_key": prompt_key(edit["subject"]),
#             "relation_id": edit["relation_id"],
#         })
#     return edit_subject_trie
#
#
# def find_edit_subject_matches(
#     prompt: str, edit_subject_trie: dict[str, Any]
# ) -> list[dict[str, Any]]:
#     words = get_words(prompt)
#     matches = []
#     for start in range(len(words)):
#         node = edit_subject_trie
#         for end in range(start, len(words)):
#             node = node.get(words[end])
#             if node is None:
#                 break
#             owners = node.get(EDIT_SUBJECT_END, [])
#             if owners:
#                 matches.extend(owners)
#     unique = {}
#     for match in matches:
#         unique[(match["edit_subject_key"], match["relation_id"])] = match
#     return list(unique.values())
#
#
# def contains_edit_subject(
#     prompt: str, relation_id: str, edit_subject_trie: dict[str, Any]
# ) -> bool:
#     return any(
#         match["relation_id"] == relation_id
#         for match in find_edit_subject_matches(prompt, edit_subject_trie)
#     )
#
#
# def build_global_entries(records) -> list[dict[str, Any]]:
#     entries = []
#     for record_index, record in enumerate(records):
#         edit = record["edit"]
#         positive_answer = edit["answer"]["new"]["str"]
#         subject_key = prompt_key(edit["subject"])
#         entries.append({
#             "entry_id": f"{record_index}:edit",
#             "record_index": record_index,
#             "role": "edit",
#             "item_index": None,
#             "subject": edit["subject"],
#             "subject_key": subject_key,
#             "relation_id": edit["relation_id"],
#             "prompt": edit["prompt"],
#             "answer": positive_answer,
#         })
#         for item_index, prompt in enumerate(record["generality"]):
#             entries.append({
#                 "entry_id": f"{record_index}:generality:{item_index}",
#                 "record_index": record_index,
#                 "role": "generality",
#                 "item_index": item_index,
#                 "subject": edit["subject"],
#                 "subject_key": subject_key,
#                 "relation_id": edit["relation_id"],
#                 "prompt": prompt,
#                 "answer": positive_answer,
#             })
#     return entries
#
#
# def build_global_groups(records) -> list[dict[str, Any]]:
#     groups = defaultdict(list)
#     for entry in build_global_entries(records):
#         groups[entry["subject_key"]].append(entry)
#     candidates = []
#     for entries in groups.values():
#         if len({prompt_key(entry["answer"]) for entry in entries}) <= 1:
#             continue
#         candidates.append({
#             "group_id": len(candidates),
#             "subject": entries[0]["subject"],
#             "entries": entries,
#         })
#     return candidates
#
#
# def build_global_audit_prompt(group: dict[str, Any]) -> str:
#     payload = json.dumps(group, ensure_ascii=False, indent=2)
#     return (
#         "Audit CounterFact prompts that mention the same subject.\n"
#         "Find sets of entries that ask the same underlying fact/relation but carry different expected answers.\n"
#         "Do not mark entries that genuinely ask different relations. For example, birthplace and citizenship, or native language and any spoken language, are different relations.\n"
#         "The entries here are edit and generality prompts describing counterfactual positive facts.\n"
#         "Return JSON only with exactly two keys: conflict_sets and reason.\n"
#         "conflict_sets must be a list of lists of entry_id strings. Every set must contain at least two entries with different answers.\n"
#         "Return an empty conflict_sets list when the differing answers belong to genuinely different facts.\n\n"
#         f"GROUP:\n{payload}"
#     )
#
#
# def parse_global_audit_response(text: str, group: dict[str, Any]) -> list[list[str]]:
#     response = text.strip().lstrip("\ufeff")
#     fenced = re.fullmatch(r"```(?:json)?\s*(.*?)\s*```", response, re.I | re.S)
#     if fenced:
#         response = fenced.group(1).strip()
#     payload = json.loads(response)
#     if set(payload) != {"conflict_sets", "reason"}:
#         raise ValueError("global audit output keys are invalid")
#     if not isinstance(payload["conflict_sets"], list) or not isinstance(payload["reason"], str):
#         raise ValueError("global audit output types are invalid")
#     entries = {entry["entry_id"]: entry for entry in group["entries"]}
#     conflicts = []
#     for conflict in payload["conflict_sets"]:
#         if not isinstance(conflict, list) or len(conflict) < 2:
#             raise ValueError("each conflict set must contain at least two entry IDs")
#         if any(entry_id not in entries for entry_id in conflict):
#             raise ValueError("global audit returned an unknown entry ID")
#         if len({prompt_key(entries[entry_id]["answer"]) for entry_id in conflict}) < 2:
#             raise ValueError("a conflict set does not contain different answers")
#         conflicts.append(list(dict.fromkeys(conflict)))
#     return conflicts
#
#
# async def audit_global_groups(
#     groups: list[dict[str, Any]],
#     config: dict[str, Any],
# ) -> tuple[list[list[str]], list[dict[str, Any]]]:
#     if not groups or not config.get("enabled", True):
#         return [], []
#     max_concurrency = config.get("max_concurrency", 8)
#     max_retries = config.get("max_retries", 2)
#     retry_delay = config.get("retry_delay", 2.0)
#     if max_concurrency <= 0 or max_retries < 0 or retry_delay < 0:
#         raise ValueError("invalid global audit concurrency/retry configuration")
#
#     client = AsyncOpenAI(
#         base_url=config.get("base_url"),
#         api_key=config.get("api_key"),
#     )
#     queue = asyncio.Queue()
#     for group in groups:
#         queue.put_nowait(group)
#     conflicts = []
#     failures = []
#     progress = tqdm(total=len(groups), desc="Auditing global prompt conflicts")
#
#     async def worker() -> None:
#         while True:
#             try:
#                 group = queue.get_nowait()
#             except asyncio.QueueEmpty:
#                 return
#             last_error = None
#             try:
#                 for attempt in range(max_retries + 1):
#                     try:
#                         completion = await client.chat.completions.create(
#                             model=config["model"],
#                             messages=[
#                                 {
#                                     "role": "system",
#                                     "content": "You audit semantic conflicts in benchmark prompts.",
#                                 },
#                                 {
#                                     "role": "user",
#                                     "content": build_global_audit_prompt(group),
#                                 },
#                             ],
#                             **config.get("request_config", {}),
#                         )
#                         content = completion.choices[0].message.content
#                         conflicts.extend(parse_global_audit_response(content, group))
#                         last_error = None
#                         break
#                     except Exception as exc:
#                         last_error = repr(exc)
#                         if attempt < max_retries:
#                             await asyncio.sleep(retry_delay * (attempt + 1))
#                 if last_error is not None:
#                     failures.append({"group": group, "error": last_error})
#             finally:
#                 queue.task_done()
#                 progress.update(1)
#
#     tasks = [
#         asyncio.create_task(worker())
#         for _ in range(min(max_concurrency, len(groups)))
#     ]
#     try:
#         await asyncio.gather(*tasks)
#     finally:
#         for task in tasks:
#             if not task.done():
#                 task.cancel()
#         await asyncio.gather(*tasks, return_exceptions=True)
#         progress.close()
#         await client.close()
#     failures.sort(key=lambda item: item["group"]["group_id"])
#     return conflicts, failures
#
#
# def finalize_dataset(
#     records,
#     semantic_conflicts: list[list[str]] | tuple[()] = (),
# ) -> tuple[Dataset, dict[str, Any]]:
#     positive_answers = defaultdict(set)
#     positive_owners = defaultdict(set)
#     locality_answers = defaultdict(set)
#     for record_index, record in enumerate(records):
#         answer = record["edit"]["answer"]["new"]["id"]
#         for prompt in [record["edit"]["prompt"], *record["generality"]]:
#             key = prompt_key(prompt)
#             positive_answers[key].add(answer)
#             positive_owners[key].add(record_index)
#         for item in record["locality"]:
#             locality_answers[prompt_key(item["prompt"])].add(
#                 item["answer"]["id"]
#             )
#
#     conflicting_positive = {
#         key for key, answers in positive_answers.items()
#         if len(answers) > 1
#     }
#     conflicting_any = {
#         key
#         for key in set(positive_answers) | set(locality_answers)
#         if len(positive_answers.get(key, set()) | locality_answers.get(key, set())) > 1
#     }
#     exact_drop_records = {
#         record_index
#         for key in conflicting_positive
#         for record_index in positive_owners[key]
#     }
#
#     semantic_drop_records = {
#         int(entry_id.split(":", 1)[0])
#         for conflict in semantic_conflicts
#         for entry_id in conflict
#     }
#
#     edit_subject_trie = build_edit_subject_trie(records)
#     old_prompt_style = re.compile(r",\s*(?:the|a|an)?\s*$", re.I)
#     seen_edits = set()
#     final_rows = []
#     report = {
#         "input_rows": len(records),
#         "conflicting_positive_prompt_keys": len(conflicting_positive),
#         "conflicting_prompt_keys": len(conflicting_any),
#         "exact_conflict_records": len(exact_drop_records),
#         "semantic_conflict_records": len(semantic_drop_records),
#         "duplicate_edit_rows": 0,
#         "old_prompt_style_rows": 0,
#         "removed_generality": 0,
#         "removed_locality": 0,
#         "empty_generality_rows": 0,
#         "empty_locality_rows": 0,
#         "invalid_generality_count_rows": 0,
#         "invalid_locality_count_rows": 0,
#         "final_rows": 0,
#     }
#
#     for record_index, record in enumerate(records):
#         if record_index in exact_drop_records or record_index in semantic_drop_records:
#             continue
#         edit_key = (
#             prompt_key(record["edit"]["prompt"]),
#             record["edit"]["answer"]["new"]["id"],
#         )
#         if edit_key in seen_edits:
#             report["duplicate_edit_rows"] += 1
#             continue
#         seen_edits.add(edit_key)
#
#         all_prompts = [
#             record["edit"]["prompt"],
#             *record["generality"],
#             *(item["prompt"] for item in record["locality"]),
#         ]
#         if any(old_prompt_style.search(prompt) for prompt in all_prompts):
#             report["old_prompt_style_rows"] += 1
#             continue
#
#         generality = list(dict.fromkeys(record["generality"]))
#         report["removed_generality"] += len(record["generality"]) - len(generality)
#         if not generality:
#             report["empty_generality_rows"] += 1
#             continue
#         if len(generality) != 3:
#             report["invalid_generality_count_rows"] += 1
#             continue
#
#         positive_keys = {
#             prompt_key(record["edit"]["prompt"]),
#             *(prompt_key(prompt) for prompt in generality),
#         }
#         locality = []
#         seen_locality = set()
#         for item_index, item in enumerate(record["locality"]):
#             key = prompt_key(item["prompt"])
#             if (
#                 key in conflicting_any
#                 or key in positive_keys
#                 or any(owner != record_index for owner in positive_owners.get(key, set()))
#                 or contains_edit_subject(
#                     item["prompt"],
#                     record["edit"]["relation_id"],
#                     edit_subject_trie,
#                 )
#                 or key in seen_locality
#             ):
#                 report["removed_locality"] += 1
#                 continue
#             seen_locality.add(key)
#             locality.append({
#                 "prompt": item["prompt"],
#                 "answer": item["answer"],
#             })
#         if not locality:
#             report["empty_locality_rows"] += 1
#             continue
#         if len(locality) != 4:
#             report["invalid_locality_count_rows"] += 1
#             continue
#
#         final_rows.append({
#             "edit": record["edit"],
#             "generality": generality,
#             "locality": locality,
#         })
#
#     report["final_rows"] = len(final_rows)
#     return Dataset.from_list(final_rows), report

In [ ]:
# import json
# import os
# from dotenv import load_dotenv
#
#
# load_dotenv()
#
# global_audit_config = {
#     "enabled": True,
#     "base_url": "http://localhost:20128/v1",
#     "api_key": os.environ["9ROUTER_API_KEY"],
#     "model": "cx/gpt-5.6-luna(low)",
#     "max_concurrency": 50,
#     "max_retries": 2,
#     "retry_delay": 2.0,
# }
#
#
# global_groups = build_global_groups(cfact_enhanced)
# semantic_conflicts, global_audit_failures = await audit_global_groups(
#     global_groups, global_audit_config
# )
# if global_audit_failures:
#     raise RuntimeError(
#         f"Global audit failed for {len(global_audit_failures)} groups; "
#         "inspect global_audit_failures before finalizing"
#     )
# cfact_final, final_report = finalize_dataset(
#     cfact_enhanced, semantic_conflicts
# )
# final_report["global_groups"] = len(global_groups)
# final_report["semantic_conflict_sets"] = len(semantic_conflicts)
# final_report["global_audit_failures"] = len(global_audit_failures)
#
# final_path = "../data/datasets/counterfact/final"
# cfact_final.save_to_disk(final_path)
# with open(f"{final_path}/report.json", "w", encoding="utf-8") as file:
#     json.dump(final_report, file, ensure_ascii=False, indent=2)
# with open(f"{final_path}/global_audit_failures.json", "w", encoding="utf-8") as file:
#     json.dump(global_audit_failures, file, ensure_ascii=False, indent=2)
# final_report

In [ ]:
# from datasets import load_from_disk
#
#
# cfact = load_from_disk("../data/datasets/counterfact/final")
# cfact

## MQuAKE

### Raw Snapshot

In [ ]:
# from datasets import load_dataset


# mquake_raw = load_dataset("henryzhongsc/MQuAKE-Remastered")
# mquake_raw.save_to_disk("../data/datasets/mquake/raw")

In [ ]:
# from datasets import load_from_disk


# mquake_raw = load_from_disk("../data/datasets/mquake/raw")
# mquake_raw

### Merge And Validation

In [ ]:
# from datasets import Dataset


# CONFIG_ORDER = ["CF9k", "CF3k", "CF6334", "T"]


# def normalize_key(text: str) -> str:
#     return " ".join(str(text or "").casefold().strip().split())


# def get_edit_key(rewrite: dict) -> tuple[str, str]:
#     return normalize_key(rewrite["subject"]), rewrite["relation_id"]


# def get_target_key(target_id: str, target: str) -> str:
#     return target_id or normalize_key(target)


# def append_unique(values: list[str], value: str) -> None:
#     marker = normalize_key(value)
#     if marker and marker not in {normalize_key(item) for item in values}:
#         values.append(value.strip())


# def bundle_answers(answer: str, aliases: list[str]) -> list[str]:
#     values = []
#     for value in [answer, *(aliases or [])]:
#         append_unique(values, value)
#     return values


# def get_edit_position(record: dict, edit_idx: int) -> int:
#     edit_triple = tuple(record["edit_triples"][edit_idx])
#     positions = [
#         hop_idx for hop_idx, hop in enumerate(record["new_triples"])
#         if tuple(hop) == edit_triple
#     ]
#     if len(positions) == 1:
#         return positions[0]

#     positions = [
#         hop_idx for hop_idx, hop in enumerate(record["new_triples"])
#         if hop[:2] == list(edit_triple[:2])
#     ]
#     if len(positions) == 1:
#         return positions[0]

#     rewrite = record["requested_rewrite"][edit_idx]
#     positions = [
#         hop_idx
#         for hop_idx, (hop, labels) in enumerate(zip(
#             record["new_triples"],
#             record["new_triples_labeled"],
#         ))
#         if hop[1] == rewrite["relation_id"]
#         and normalize_key(labels[0]) == normalize_key(rewrite["subject"])
#         and normalize_key(labels[2]) == normalize_key(rewrite["target_new_str"])
#     ]
#     if len(positions) != 1:
#         raise ValueError(f"Expected one hop for edit {edit_idx}, got {positions}")
#     return positions[0]


# def make_hop(record: dict, hop_idx: int, state: str) -> dict:
#     triple_key = "orig_triples" if state == "true" else "new_triples"
#     label_key = f"{triple_key}_labeled"
#     hop_key = "single_hops" if state == "true" else "new_single_hops"
#     triple = record[triple_key][hop_idx]
#     labels = record[label_key][hop_idx]
#     hop = record[hop_key][hop_idx]
#     aliases = []
#     for value in [hop["answer"], *(hop["answer_alias"] or [])]:
#         if normalize_key(value) != normalize_key(labels[2]):
#             append_unique(aliases, value)

#     return {
#         "prompt": {
#             "cloze": hop["cloze"],
#             "question": hop["question"],
#         },
#         "subject": {"id": triple[0], "str": labels[0]},
#         "relation": {"id": triple[1], "str": labels[1]},
#         "answer": {
#             "id": triple[2],
#             "str": labels[2],
#             "aliases": aliases,
#         },
#     }


# def make_rewrite(record: dict, edit_idx: int, hop_idx: int, start: int, selected_idx: int) -> dict:
#     rewrite = record["requested_rewrite"][edit_idx]
#     edit_triple = record["edit_triples"][edit_idx]
#     relation = record["new_triples_labeled"][hop_idx][1]
#     return {
#         "position": hop_idx - start,
#         "selected": edit_idx == selected_idx,
#         "prompt": rewrite["prompt"].format(rewrite["subject"]),
#         "question": rewrite["question"],
#         "subject": {"id": edit_triple[0], "str": rewrite["subject"]},
#         "relation": {"id": rewrite["relation_id"], "str": relation},
#         "answer": {
#             "true": {"id": rewrite["target_true_id"], "str": rewrite["target_true_str"]},
#             "new": {"id": rewrite["target_new_id"], "str": rewrite["target_new_str"]},
#         },
#     }


# true_targets_by_key = {}
# for config_name in CONFIG_ORDER:
#     for record in mquake_raw[config_name]:
#         for rewrite in record["requested_rewrite"]:
#             key = get_edit_key(rewrite)
#             true_targets_by_key.setdefault(key, set()).add(
#                 get_target_key(rewrite["target_true_id"], rewrite["target_true_str"])
#             )

# valid_keys = {key for key, targets in true_targets_by_key.items() if len(targets) == 1}
# merged_by_key = {}
# variants_by_key = {}

# for config_name in CONFIG_ORDER:
#     for record in mquake_raw[config_name]:
#         edit_positions = [
#             get_edit_position(record, edit_idx)
#             for edit_idx in range(len(record["requested_rewrite"]))
#         ]
#         for edit_idx, (rewrite, edit_pos) in enumerate(zip(
#             record["requested_rewrite"],
#             edit_positions,
#         )):
#             edit_key = get_edit_key(rewrite)
#             if edit_key not in valid_keys:
#                 continue

#             edit = merged_by_key.setdefault(edit_key, {
#                 "edit": {
#                     "prompts": [],
#                     "questions": [],
#                     "subject": {"ids": [], "str": rewrite["subject"]},
#                     "relation": {"id": rewrite["relation_id"], "strs": []},
#                     "answer": {
#                         "true": {
#                             "id": rewrite["target_true_id"],
#                             "strs": [],
#                         },
#                     },
#                 },
#                 "variants": [],
#             })
#             edit_data = edit["edit"]
#             append_unique(edit_data["subject"]["ids"], record["edit_triples"][edit_idx][0])
#             append_unique(edit_data["relation"]["strs"], record["new_triples_labeled"][edit_pos][1])
#             append_unique(edit_data["answer"]["true"]["strs"], rewrite["target_true_str"])
#             append_unique(edit_data["prompts"], rewrite["prompt"].format(rewrite["subject"]))
#             append_unique(edit_data["questions"], rewrite["question"])

#             new_target_key = get_target_key(rewrite["target_new_id"], rewrite["target_new_str"])
#             variant_key = (*edit_key, new_target_key)
#             if variant_key not in variants_by_key:
#                 variant = {
#                     "answer": {
#                         "new": {
#                             "id": rewrite["target_new_id"],
#                             "strs": [],
#                         },
#                     },
#                     "portability": [],
#                 }
#                 variants_by_key[variant_key] = variant
#                 edit["variants"].append(variant)
#             variant = variants_by_key[variant_key]
#             append_unique(variant["answer"]["new"]["strs"], rewrite["target_new_str"])
#             selected_hop = record["new_single_hops"][edit_pos]
#             for answer in [selected_hop["answer"], *(selected_hop["answer_alias"] or [])]:
#                 append_unique(variant["answer"]["new"]["strs"], answer)

#             windows = []
#             if edit_pos > 0:
#                 windows.append(("left", edit_pos - 1))
#             if edit_pos + 1 < len(record["new_triples"]):
#                 windows.append(("right", edit_pos))

#             for direction, start in windows:
#                 stop = start + 2
#                 variant["portability"].append({
#                     "direction": direction,
#                     "selected_position": edit_pos - start,
#                     "source": {
#                         "config": config_name,
#                         "case_id": record["case_id"],
#                         "hop_count": len(record["orig_triples"]),
#                         "questions": list(record["questions"]),
#                         "answers": {
#                             "true": bundle_answers(
#                                 record["answer"],
#                                 [*(record["answer_alias"] or []), *(record["answer_extended"] or [])],
#                             ),
#                             "new": bundle_answers(record["new_answer"], record["new_answer_alias"] or []),
#                         },
#                     },
#                     "hops": {
#                         "true": [make_hop(record, hop_idx, "true") for hop_idx in range(start, stop)],
#                         "new": [make_hop(record, hop_idx, "new") for hop_idx in range(start, stop)],
#                     },
#                     "rewrites": [
#                         make_rewrite(record, other_idx, other_pos, start, edit_idx)
#                         for other_idx, other_pos in enumerate(edit_positions)
#                         if start <= other_pos < stop
#                     ],
#                 })

# mquake_merged = Dataset.from_list(list(merged_by_key.values()))
# mquake_merged.save_to_disk("../data/datasets/mquake/merged")

In [ ]:
# from datasets import load_from_disk


# mquake_merged = load_from_disk("../data/datasets/mquake/merged")
# mquake_merged

### Cleaning, Restoration And Conflict Selection

In [ ]:
# import copy
# import json
# from collections import Counter, defaultdict
# from datasets import Dataset
# import numpy as np
# from scipy.optimize import Bounds, LinearConstraint, milp
# from scipy.sparse import coo_matrix


# def normalize_text(text) -> str:
#     return " ".join(str(text or "").casefold().strip().split())


# def unique_texts(values: list[str]) -> list[str]:
#     output, seen = [], set()
#     for value in values:
#         marker = normalize_text(value)
#         if marker and marker not in seen:
#             seen.add(marker)
#             output.append(str(value).strip())
#     return output


# def entity_key(entity: dict) -> tuple[str, str]:
#     entity_id = entity.get("id")
#     return ("id", entity_id) if entity_id else ("text", normalize_text(entity.get("str")))


# def entities_match(left: dict, right: dict) -> bool:
#     left_id, right_id = left.get("id"), right.get("id")
#     if left_id and right_id:
#         return left_id == right_id
#     return normalize_text(left.get("str")) == normalize_text(right.get("str"))


# def answer_texts(answer: dict) -> set[str]:
#     values = [answer.get("str"), *(answer.get("aliases") or [])]
#     values.extend(entity_names.get(answer.get("id"), []))
#     return {marker for value in values if (marker := normalize_text(value))}


# def answers_overlap(left: dict, right: dict) -> bool:
#     return entities_match(left, right) or bool(answer_texts(left) & answer_texts(right))


# def bundle_answer(answer: dict) -> dict:
#     values = unique_texts([*answer["strs"], *entity_names.get(answer["id"], [])])
#     return {"id": answer["id"], "str": values[0], "aliases": values[1:]}


# def chain_is_continuous(hops: list[dict]) -> bool:
#     return len(hops) == 2 and entities_match(hops[0]["answer"], hops[1]["subject"])


# def rewrite_matches_hop(hop: dict, rewrite: dict, state: str) -> bool:
#     return (
#         hop["relation"]["id"] == rewrite["relation"]["id"]
#         and entities_match(hop["subject"], rewrite["subject"])
#         and entities_match(hop["answer"], rewrite["answer"][state])
#     )


# def hop_key(hop: dict) -> tuple:
#     return (entity_key(hop["subject"]), hop["relation"]["id"], entity_key(hop["answer"]))


# def window_key(target: dict, window: dict) -> tuple:
#     return (
#         entity_key(target),
#         window["selected_position"],
#         tuple(hop_key(hop) for hop in window["hops"]["true"]),
#         tuple(hop_key(hop) for hop in window["hops"]["new"]),
#     )


# def query_key(window: dict) -> tuple:
#     hops = window["hops"]["new"]
#     return (entity_key(hops[0]["subject"]), hops[0]["relation"]["id"], hops[1]["relation"]["id"])


# def clean_window(window: dict) -> tuple[dict | None, str | None]:
#     cleaned = copy.deepcopy(window)
#     true_hops, new_hops = cleaned["hops"]["true"], cleaned["hops"]["new"]
#     for hop in [*true_hops, *new_hops]:
#         answer = hop["answer"]
#         names = unique_texts([answer["str"], *answer["aliases"], *entity_names.get(answer["id"], [])])
#         answer["str"], answer["aliases"] = names[0], names[1:]
#     position = cleaned["selected_position"]
#     selected = [rewrite for rewrite in cleaned["rewrites"] if rewrite["selected"]]
#     neighbors = [rewrite for rewrite in cleaned["rewrites"] if not rewrite["selected"]]
#     if len(true_hops) != 2 or len(new_hops) != 2:
#         return None, "invalid_hop_count"
#     if position not in (0, 1):
#         return None, "invalid_selected_position"
#     if len(selected) != 1:
#         return None, "invalid_selected_rewrite"
#     if selected[0]["position"] != position:
#         return None, "selected_position_mismatch"
#     if len(neighbors) > 1 or (neighbors and neighbors[0]["position"] != 1 - position):
#         return None, "invalid_neighbor_rewrite"
#     if not chain_is_continuous(true_hops):
#         return None, "broken_true_chain"
#     if not rewrite_matches_hop(true_hops[position], selected[0], "true"):
#         return None, "selected_true_hop_mismatch"
#     if not rewrite_matches_hop(new_hops[position], selected[0], "new"):
#         return None, "selected_new_hop_mismatch"
#     if true_hops[1 - position]["relation"]["id"] != new_hops[1 - position]["relation"]["id"]:
#         return None, "neighbor_relation_changed"
#     if position == 0 and neighbors:
#         recovered = neighbors[0]["answer"]["true"]
#         recovered_names = unique_texts([recovered["str"], *entity_names.get(recovered["id"], [])])
#         new_hops[1]["answer"] = {"id": recovered["id"], "str": recovered_names[0], "aliases": recovered_names[1:]}
#     elif position == 1:
#         new_hops[0] = copy.deepcopy(true_hops[0])
#     if not chain_is_continuous(new_hops):
#         return None, "failed_neighbor_recovery"
#     true_answer, new_answer = true_hops[-1]["answer"], new_hops[-1]["answer"]
#     if answers_overlap(true_answer, new_answer):
#         return None, "unchanged_final_answer"
#     source = cleaned.pop("source")
#     questions = unique_texts(source["questions"])
#     if not questions:
#         return None, "empty_questions"
#     cleaned["answers"] = {
#         "true": unique_texts([true_answer["str"], *true_answer["aliases"]]),
#         "new": unique_texts([new_answer["str"], *new_answer["aliases"]]),
#     }
#     cleaned["sources"] = [{"config": source["config"], "case_id": source["case_id"], "hop_count": source["hop_count"], "questions": questions, "answers": source["answers"]}]
#     return cleaned, None


# def merge_sources(target: dict, duplicate: dict) -> None:
#     sources = {(source["config"], source["case_id"]): source for source in target["sources"]}
#     for source in duplicate["sources"]:
#         key = source["config"], source["case_id"]
#         if key in sources:
#             sources[key]["questions"] = unique_texts([*sources[key]["questions"], *source["questions"]])
#         else:
#             target["sources"].append(source)
#             sources[key] = source


# def add_constraint(coefficients: dict[int, float], lower: float = -np.inf, upper: float = np.inf) -> None:
#     constraint_index = len(constraint_lower)
#     for variable, coefficient in coefficients.items():
#         if coefficient:
#             constraint_rows.append(constraint_index)
#             constraint_columns.append(variable)
#             constraint_values.append(coefficient)
#     constraint_lower.append(lower)
#     constraint_upper.append(upper)


# merged_rows = list(mquake_merged)
# entity_names = defaultdict(list)
# for row in merged_rows:
#     edit = row["edit"]
#     for subject_id in edit["subject"]["ids"]:
#         if subject_id:
#             entity_names[subject_id] = unique_texts([*entity_names[subject_id], edit["subject"]["str"]])
#     true_answer = edit["answer"]["true"]
#     entity_names[true_answer["id"]] = unique_texts([*entity_names[true_answer["id"]], *true_answer["strs"]])
#     for variant in row["variants"]:
#         new_answer = variant["answer"]["new"]
#         entity_names[new_answer["id"]] = unique_texts([*entity_names[new_answer["id"]], *new_answer["strs"]])
#         for window in variant["portability"]:
#             for hop in [*window["hops"]["true"], *window["hops"]["new"]]:
#                 for entity in (hop["subject"], hop["answer"]):
#                     values = [entity["str"], *(entity.get("aliases") or [])]
#                     entity_names[entity["id"]] = unique_texts([*entity_names[entity["id"]], *values])
#             for rewrite in window["rewrites"]:
#                 entity_names[rewrite["subject"]["id"]] = unique_texts([*entity_names[rewrite["subject"]["id"]], rewrite["subject"]["str"]])
#                 for answer in rewrite["answer"].values():
#                     entity_names[answer["id"]] = unique_texts([*entity_names[answer["id"]], answer["str"]])
# clean_reasons = Counter()
# edit_rows, true_answers = {}, {}
# for row_index, row in enumerate(merged_rows):
#     subject_ids = {subject_id for subject_id in row["edit"]["subject"]["ids"] if subject_id}
#     if len(subject_ids) != 1:
#         clean_reasons["ambiguous_edit_subject_id"] += 1
#         continue
#     edit_key = (("id", next(iter(subject_ids))), row["edit"]["relation"]["id"])
#     if edit_key in edit_rows:
#         raise ValueError(f"Duplicate MQuAKE edit key: {edit_key}")
#     edit_rows[edit_key] = row_index
#     true_answers[edit_key] = bundle_answer(row["edit"]["answer"]["true"])

# variants, windows = [], []
# for edit_key, row_index in edit_rows.items():
#     row = merged_rows[row_index]
#     true_answer = true_answers[edit_key]
#     for variant_index, variant in enumerate(row["variants"]):
#         new_answer = bundle_answer(variant["answer"]["new"])
#         if answers_overlap(true_answer, new_answer):
#             clean_reasons["textually_unchanged_edit_variant"] += 1
#             continue
#         candidate = {
#             "row_index": row_index,
#             "variant_index": variant_index,
#             "edit_key": edit_key,
#             "target": new_answer,
#             "answer": variant["answer"],
#             "window_indices": [],
#         }
#         structural_windows = {}
#         for window in variant["portability"]:
#             cleaned_window, reason = clean_window(window)
#             if reason:
#                 clean_reasons[reason] += 1
#                 continue
#             dependency_hop = cleaned_window["hops"]["new"][1 - cleaned_window["selected_position"]]
#             dependency_key = (entity_key(dependency_hop["subject"]), dependency_hop["relation"]["id"])
#             if dependency_key in true_answers and not entities_match(dependency_hop["answer"], true_answers[dependency_key]):
#                 clean_reasons["dependency_true_answer_mismatch"] += 1
#                 continue
#             cleaned_window["dependency_key"] = dependency_key if dependency_key in edit_rows else None
#             key = window_key(new_answer, cleaned_window)
#             if key in structural_windows:
#                 merge_sources(structural_windows[key], cleaned_window)
#                 clean_reasons["duplicate_window"] += 1
#             else:
#                 structural_windows[key] = cleaned_window
#         if structural_windows:
#             variant_id = len(variants)
#             for cleaned_window in structural_windows.values():
#                 cleaned_window["variant_id"] = variant_id
#                 candidate["window_indices"].append(len(windows))
#                 windows.append(cleaned_window)
#             variants.append(candidate)
#         else:
#             clean_reasons["variant_without_windows"] += 1

# variants_by_edit = defaultdict(list)
# for variant_id, variant in enumerate(variants):
#     variants_by_edit[variant["edit_key"]].append(variant_id)
# query_answers = defaultdict(lambda: defaultdict(list))
# for window_id, window in enumerate(windows):
#     answer_key = entity_key(window["hops"]["new"][-1]["answer"])
#     query_answers[query_key(window)][answer_key].append(window_id)

# variant_count, window_count = len(variants), len(windows)
# query_group_keys = [
#     (query, answer)
#     for query, answer_groups in query_answers.items()
#     if len(answer_groups) > 1
#     for answer in answer_groups
# ]
# query_group_variables = {
#     key: variant_count + window_count + index
#     for index, key in enumerate(query_group_keys)
# }
# variable_count = variant_count + window_count + len(query_group_variables)
# constraint_rows, constraint_columns, constraint_values = [], [], []
# constraint_lower, constraint_upper = [], []

# for variant_ids in variants_by_edit.values():
#     add_constraint({variant_id: 1.0 for variant_id in variant_ids}, upper=1.0)
# for variant_id, variant in enumerate(variants):
#     window_variables = [variant_count + window_id for window_id in variant["window_indices"]]
#     add_constraint({variant_id: 1.0, **{variable: -1.0 for variable in window_variables}}, upper=0.0)
#     for variable in window_variables:
#         add_constraint({variable: 1.0, variant_id: -1.0}, upper=0.0)
# for window_id, window in enumerate(windows):
#     dependency_key = window["dependency_key"]
#     if dependency_key is None:
#         continue
#     expected_answer = window["hops"]["new"][1 - window["selected_position"]]["answer"]
#     incompatible = {
#         variant_id: 1.0
#         for variant_id in variants_by_edit.get(dependency_key, [])
#         if not entities_match(variants[variant_id]["target"], expected_answer)
#     }
#     if incompatible:
#         add_constraint({variant_count + window_id: 1.0, **incompatible}, upper=1.0)
# for query, answer_groups in query_answers.items():
#     if len(answer_groups) <= 1:
#         continue
#     group_variables = {query_group_variables[(query, answer)]: 1.0 for answer in answer_groups}
#     add_constraint(group_variables, upper=1.0)
#     for answer, window_ids in answer_groups.items():
#         group_variable = query_group_variables[(query, answer)]
#         window_variables = [variant_count + window_id for window_id in window_ids]
#         add_constraint({group_variable: 1.0, **{variable: -1.0 for variable in window_variables}}, upper=0.0)
#         for variable in window_variables:
#             add_constraint({variable: 1.0, group_variable: -1.0}, upper=0.0)


# def solve_selection(objective: np.ndarray, fixed_edits: int | None = None, fixed_windows: int | None = None):
#     lower, upper = list(constraint_lower), list(constraint_upper)
#     rows, columns, values = list(constraint_rows), list(constraint_columns), list(constraint_values)
#     if fixed_edits is not None:
#         constraint_index = len(lower)
#         rows.extend([constraint_index] * variant_count)
#         columns.extend(range(variant_count))
#         values.extend([1.0] * variant_count)
#         lower.append(float(fixed_edits))
#         upper.append(float(fixed_edits))
#     if fixed_windows is not None:
#         constraint_index = len(lower)
#         rows.extend([constraint_index] * window_count)
#         columns.extend(range(variant_count, variant_count + window_count))
#         values.extend([1.0] * window_count)
#         lower.append(float(fixed_windows))
#         upper.append(float(fixed_windows))
#     matrix = coo_matrix((values, (rows, columns)), shape=(len(lower), variable_count)).tocsr()
#     return milp(
#         objective,
#         integrality=np.ones(variable_count),
#         bounds=Bounds(np.zeros(variable_count), np.ones(variable_count)),
#         constraints=LinearConstraint(matrix, np.asarray(lower), np.asarray(upper)),
#         options={"presolve": True, "mip_rel_gap": 0.0},
#     )


# edit_objective = np.zeros(variable_count)
# edit_objective[:variant_count] = -1.0
# edit_solution = solve_selection(edit_objective)
# if not edit_solution.success:
#     raise RuntimeError(edit_solution.message)
# max_edits = round(edit_solution.x[:variant_count].sum())
# window_objective = np.zeros(variable_count)
# window_objective[variant_count:variant_count + window_count] = -1.0
# window_solution = solve_selection(window_objective, fixed_edits=max_edits)
# if not window_solution.success:
#     raise RuntimeError(window_solution.message)
# max_windows = round(window_solution.x[variant_count:variant_count + window_count].sum())
# stable_objective = np.zeros(variable_count)
# stable_objective[:variant_count + window_count] = np.arange(1, variant_count + window_count + 1)
# selection = solve_selection(stable_objective, max_edits, max_windows)
# if not selection.success:
#     raise RuntimeError(selection.message)

# selected_variants = {variant_id for variant_id in range(variant_count) if selection.x[variant_id] > 0.5}
# selected_windows = {window_id for window_id in range(window_count) if selection.x[variant_count + window_id] > 0.5}
# clean_windows_by_direction = Counter()
# clean_sources_by_config = Counter()
# clean_source_occurrences = 0
# clean_rows = []
# for variant_id in sorted(selected_variants):
#     variant = variants[variant_id]
#     portability = []
#     for window_id in variant["window_indices"]:
#         if window_id not in selected_windows:
#             continue
#         window = copy.deepcopy(windows[window_id])
#         window.pop("variant_id")
#         window.pop("dependency_key")
#         clean_windows_by_direction[window["direction"]] += 1
#         clean_source_occurrences += len(window["sources"])
#         for source in window["sources"]:
#             clean_sources_by_config[source["config"]] += 1
#         portability.append(window)
#     clean_edit = copy.deepcopy(merged_rows[variant["row_index"]]["edit"])
#     clean_true_answer = bundle_answer(clean_edit["answer"]["true"])
#     clean_edit["answer"]["true"]["strs"] = [clean_true_answer["str"], *clean_true_answer["aliases"]]
#     clean_variant_answer = copy.deepcopy(variant["answer"])
#     clean_variant_answer["new"]["strs"] = [variant["target"]["str"], *variant["target"]["aliases"]]
#     clean_rows.append({
#         "edit": clean_edit,
#         "variants": [{"answer": clean_variant_answer, "portability": portability}],
#     })

# selected_targets = {variants[variant_id]["edit_key"]: variants[variant_id]["target"] for variant_id in selected_variants}
# query_validation = defaultdict(set)
# structural_validation = set()
# selected_dependency_windows = 0
# for window_id in selected_windows:
#     window = windows[window_id]
#     assert window["variant_id"] in selected_variants
#     selected_target = variants[window["variant_id"]]["target"]
#     assert entities_match(selected_target, window["hops"]["new"][window["selected_position"]]["answer"])
#     assert chain_is_continuous(window["hops"]["true"])
#     assert chain_is_continuous(window["hops"]["new"])
#     assert not answers_overlap(window["hops"]["true"][-1]["answer"], window["hops"]["new"][-1]["answer"])
#     dependency_key = window["dependency_key"]
#     if dependency_key in selected_targets:
#         selected_dependency_windows += 1
#         expected = window["hops"]["new"][1 - window["selected_position"]]["answer"]
#         assert entities_match(selected_targets[dependency_key], expected)
#     query_validation[query_key(window)].add(entity_key(window["hops"]["new"][-1]["answer"]))
#     signature = (variants[window["variant_id"]]["edit_key"], window_key(variants[window["variant_id"]]["target"], window))
#     assert signature not in structural_validation
#     structural_validation.add(signature)
# assert all(len(answers) == 1 for answers in query_validation.values())
# assert len(clean_rows) == max_edits >= 5000

# mquake_clean = Dataset.from_list(clean_rows)
# mquake_clean.save_to_disk("../data/datasets/mquake/clean")
# with open("../data/datasets/mquake/clean/report.json", "w", encoding="utf-8") as file:
#     json.dump({
#         "merged": {
#             "edits": len(mquake_merged),
#             "variants": sum(len(row["variants"]) for row in mquake_merged),
#             "windows": sum(len(variant["portability"]) for row in mquake_merged for variant in row["variants"]),
#         },
#         "local": {
#             "candidate_edits": len(variants_by_edit),
#             "candidate_variants": variant_count,
#             "structural_windows": window_count,
#             "reasons": dict(clean_reasons),
#         },
#         "graph": {
#             "variables": variable_count,
#             "constraints": len(constraint_lower),
#             "conflicting_query_keys": sum(len(groups) > 1 for groups in query_answers.values()),
#             "maximum_edits": max_edits,
#             "maximum_windows_at_edit_optimum": max_windows,
#             "solver_message": str(selection.message),
#         },
#         "clean": {
#             "edits": len(mquake_clean),
#             "variants": sum(len(row["variants"]) for row in mquake_clean),
#             "windows": sum(len(variant["portability"]) for row in mquake_clean for variant in row["variants"]),
#             "windows_by_direction": dict(clean_windows_by_direction),
#             "source_occurrences": clean_source_occurrences,
#             "sources_by_config": dict(clean_sources_by_config),
#             "selected_dependency_windows": selected_dependency_windows,
#             "validation": {
#                 "unique_edit_keys": True,
#                 "one_variant_per_edit": True,
#                 "continuous_two_hop_chains": True,
#                 "selected_edit_alignment": True,
#                 "no_final_answer_overlap": True,
#                 "no_selected_dependency_conflicts": True,
#                 "one_answer_per_semantic_query": True,
#                 "no_structural_duplicates": True,
#             },
#         },
#     }, file, ensure_ascii=False, indent=2)

In [ ]:
# from datasets import load_from_disk


# mquake_clean = load_from_disk("../data/datasets/mquake/clean")
# mquake_clean

### Uniform Selection

In [ ]:
# import hashlib
# import json
# import re
# from collections import Counter, defaultdict
# from datasets import Dataset


# def normalize_uniform_text(text: str) -> str:
#     return " ".join(str(text or "").casefold().strip().split())


# def uniform_texts(values: list[str]) -> set[str]:
#     return {marker for value in values if (marker := normalize_uniform_text(value))}


# def uniform_words(text: str) -> list[str]:
#     return re.findall(r"[0-9a-z]+", str(text or "").casefold())


# def uniform_contains_phrase(text: str, phrase: str) -> bool:
#     text_words = uniform_words(text)
#     phrase_words = uniform_words(phrase)
#     return bool(phrase_words) and any(
#         text_words[start:start + len(phrase_words)] == phrase_words
#         for start in range(len(text_words) - len(phrase_words) + 1)
#     )


# def uniform_window_rejection(window: dict) -> str | None:
#     new_hops = window["hops"]["new"]
#     final_answer = new_hops[-1]["answer"]
#     if final_answer["id"] in {hop["subject"]["id"] for hop in new_hops}:
#         return "final answer repeats a chain subject"
#     if uniform_contains_phrase(new_hops[0]["subject"]["str"], final_answer["str"]):
#         return "final answer appears in the chain-start subject"
#     return None


# LOCATION_RELATIONS = {"P20", "P159", "P740", "P937"}


# def get_rewrite_reasons(record: dict, source_hop_count: int) -> list[str]:
#     portability = record["portability"]
#     hop1, hop2 = portability["hops"]["new"]
#     relation1 = hop1["relation"]["id"]
#     relation2 = hop2["relation"]["id"]
#     prompts = [prompt.casefold() for prompt in portability["prompts"]]
#     reasons = []

#     if source_hop_count != 2:
#         reasons.append("source_chain_length")
#     if relation2 == "P136":
#         reasons.append("generic_genre")
#     if (relation1, relation2) == ("P175", "P1037"):
#         reasons.append("performer_manager")
#     if relation2 == "P19":
#         reasons.append("birthplace_wording")
#     if relation2 == "P407" and relation1 in {"P108", "P178"}:
#         reasons.append("language_context")
#     if relation2 == "P364":
#         reasons.append("original_language")
#     if relation1 == "P1308":
#         reasons.append("officeholder_reference")
#     if relation2 in LOCATION_RELATIONS and any("city" in prompt for prompt in prompts):
#         reasons.append("location_type_assumption")
#     if (relation1, relation2) in {("P740", "P112"), ("P449", "P112")}:
#         reasons.append("founder_type_assumption")
#     return reasons


# def source_alignment(source: dict, window: dict) -> tuple[int, int]:
#     source_true = uniform_texts(source["answers"]["true"])
#     source_new = uniform_texts(source["answers"]["new"])
#     window_true = uniform_texts(window["answers"]["true"])
#     window_new = uniform_texts(window["answers"]["new"])
#     true_match, new_match = bool(source_true & window_true), bool(source_new & window_new)
#     alignment = 3 if true_match and new_match else 2 if true_match else 1 if new_match else 0
#     exact_matches = int(source_true == window_true) + int(source_new == window_new)
#     return alignment, exact_matches


# def uniform_hash(value: dict) -> str:
#     payload = json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=True)
#     return hashlib.sha256(payload.encode("utf-8")).hexdigest()


# def uniform_window_hash(edit: dict, variant: dict, window: dict) -> str:
#     def hop_key(hop: dict) -> tuple[str, str, str]:
#         return hop["subject"]["id"], hop["relation"]["id"], hop["answer"]["id"]

#     return uniform_hash({
#         "edit": [edit["subject"]["ids"][0], edit["relation"]["id"]],
#         "target": variant["answer"]["new"]["id"],
#         "selected_position": window["selected_position"],
#         "true_hops": [hop_key(hop) for hop in window["hops"]["true"]],
#         "new_hops": [hop_key(hop) for hop in window["hops"]["new"]],
#     })


# def uniform_source_hash(source: dict) -> str:
#     return uniform_hash({
#         "config": source["config"],
#         "case_id": source["case_id"],
#         "questions": source["questions"],
#     })


# def make_uniform_hop(hop: dict) -> dict:
#     answers = [hop["answer"]["str"], *hop["answer"]["aliases"]]
#     return {
#         "prompt": hop["prompt"]["cloze"],
#         "subject": hop["subject"],
#         "relation": hop["relation"],
#         "answers": {"id": hop["answer"]["id"], "str": answers},
#     }


# def direct_prompt_conflicts(selections: list[dict | None]) -> dict[str, set[str]]:
#     answers_by_prompt = defaultdict(set)
#     for candidate in selections:
#         if candidate is None or candidate["source"]["hop_count"] != 2:
#             continue
#         answer_id = candidate["window"]["hops"]["new"][-1]["answer"]["id"]
#         for prompt in candidate["question_keys"]:
#             answers_by_prompt[prompt].add(answer_id)
#     return {prompt: answers for prompt, answers in answers_by_prompt.items() if len(answers) > 1}


# def repair_direct_conflicts(
#     candidate_lists: list[list[dict]],
#     selections: list[dict | None],
# ) -> tuple[list[dict | None], list[dict]]:
#     repairs = []
#     while conflicts := direct_prompt_conflicts(selections):
#         affected = []
#         for index, candidate in enumerate(selections):
#             if candidate is None or candidate["source"]["hop_count"] != 2:
#                 continue
#             if candidate["question_keys"] & conflicts.keys():
#                 affected.append(index)

#         options = []
#         for index in affected:
#             for rank, candidate in enumerate(candidate_lists[index]):
#                 if candidate is selections[index]:
#                     continue
#                 trial = list(selections)
#                 trial[index] = candidate
#                 remaining = len(direct_prompt_conflicts(trial))
#                 if remaining < len(conflicts):
#                     options.append((remaining, rank, candidate["key"], index, candidate))

#         if options:
#             remaining, rank, _, index, candidate = min(options)
#             selections[index] = candidate
#             repairs.append({
#                 "record_index": candidate["record_index"],
#                 "action": "reselect",
#                 "candidate_rank": rank,
#                 "conflicting_prompts_before": len(conflicts),
#                 "conflicting_prompts_after": remaining,
#             })
#             continue

#         drop_options = []
#         for index in affected:
#             trial = list(selections)
#             trial[index] = None
#             remaining = len(direct_prompt_conflicts(trial))
#             if remaining < len(conflicts):
#                 drop_options.append((remaining, selections[index]["record_index"], index))
#         if not drop_options:
#             raise ValueError("Unable to resolve direct MQuAKE prompt conflicts")
#         remaining, record_index, index = min(drop_options)
#         selections[index] = None
#         repairs.append({
#             "record_index": record_index,
#             "action": "drop",
#             "conflicting_prompts_before": len(conflicts),
#             "conflicting_prompts_after": remaining,
#         })
#     return selections, repairs


# uniform_rows = []
# alignment_counts = Counter()
# hop_counts = Counter()
# direction_counts = Counter()
# source_config_counts = Counter()
# quality_ties = 0
# multi_window_edits = 0
# rejected_windows = Counter()
# dropped_edits = []
# uniform_records = []
# candidate_lists = []
# for record_index, row in enumerate(mquake_clean):
#     if len(row["variants"]) != 1:
#         raise ValueError("Every clean MQuAKE edit must have exactly one variant")
#     edit, variant = row["edit"], row["variants"][0]
#     if len(variant["portability"]) > 1:
#         multi_window_edits += 1
#     candidates = []
#     for window in variant["portability"]:
#         rejection = uniform_window_rejection(window)
#         if rejection:
#             rejected_windows[rejection] += 1
#             continue
#         window_hash = uniform_window_hash(edit, variant, window)
#         for source in window["sources"]:
#             questions = list(source["questions"])
#             question_keys = {normalize_uniform_text(question) for question in questions}
#             if len(questions) != 3 or len(question_keys) != 3 or any(not question.endswith("?") for question in questions):
#                 continue
#             alignment, exact_matches = source_alignment(source, window)
#             quality = (abs(source["hop_count"] - 2), -alignment, -exact_matches)
#             key = (*quality, window_hash, uniform_source_hash(source))
#             candidates.append({
#                 "key": key,
#                 "quality": quality,
#                 "alignment": alignment,
#                 "window": window,
#                 "source": source,
#                 "question_keys": question_keys,
#                 "record_index": record_index,
#             })
#     if not candidates:
#         dropped_edits.append({
#             "record_index": record_index,
#             "subject": edit["subject"]["str"],
#             "relation": edit["relation"]["strs"][0],
#         })
#         continue
#     candidates.sort(key=lambda candidate: candidate["key"])
#     quality_ties += sum(
#         candidate["quality"] == candidates[0]["quality"] for candidate in candidates
#     ) > 1
#     uniform_records.append((record_index, edit, variant))
#     candidate_lists.append(candidates)

# selections = [candidates[0] for candidates in candidate_lists]
# selections, conflict_repairs = repair_direct_conflicts(
#     candidate_lists,
#     selections,
# )
# assert not direct_prompt_conflicts(selections)
# for (record_index, edit, variant), candidate in zip(uniform_records, selections):
#     if candidate is None:
#         dropped_edits.append({
#             "record_index": record_index,
#             "subject": edit["subject"]["str"],
#             "relation": edit["relation"]["strs"][0],
#         })
#         continue
#     alignment, window, source = candidate["alignment"], candidate["window"], candidate["source"]
#     alignment_counts[alignment] += 1
#     hop_counts[source["hop_count"]] += 1
#     direction_counts[window["direction"]] += 1
#     source_config_counts[source["config"]] += 1
#     true_answer, new_answer = edit["answer"]["true"], variant["answer"]["new"]
#     uniform_record = {
#         "edit": {
#             "prompt": edit["prompts"][0],
#             "subject": {"id": edit["subject"]["ids"][0], "str": edit["subject"]["str"]},
#             "relation": {"id": edit["relation"]["id"], "str": edit["relation"]["strs"][0]},
#             "answer": {
#                 "true": {"id": true_answer["id"], "str": true_answer["strs"][0]},
#                 "new": {"id": new_answer["id"], "str": new_answer["strs"][0]},
#             },
#         },
#         "portability": {
#             "prompts": list(source["questions"]),
#             "answers": window["answers"],
#             "hops": {
#                 state: [make_uniform_hop(hop) for hop in window["hops"][state]]
#                 for state in ("true", "new")
#             },
#         },
#     }
#     rewrite_reasons = get_rewrite_reasons(uniform_record, source["hop_count"])
#     uniform_record["portability"]["rewrite_required"] = bool(rewrite_reasons)
#     uniform_record["portability"]["rewrite_reasons"] = rewrite_reasons
#     uniform_rows.append(uniform_record)

# rewrite_reason_counts = Counter(
#     reason
#     for row in uniform_rows
#     for reason in row["portability"]["rewrite_reasons"]
# )
# mquake_uniform = Dataset.from_list(uniform_rows)
# assert len(mquake_uniform) == len(mquake_clean) - len(dropped_edits)
# assert all(
#     row["portability"]["rewrite_required"] == bool(row["portability"]["rewrite_reasons"])
#     for row in mquake_uniform
# )
# assert all(len(row["portability"]["prompts"]) == 3 for row in mquake_uniform)
# assert all(len(row["portability"]["hops"][state]) == 2 for row in mquake_uniform for state in ("true", "new"))
# mquake_uniform.save_to_disk("../data/datasets/mquake/uniform")
# with open("../data/datasets/mquake/uniform/report.json", "w", encoding="utf-8") as file:
#     json.dump({
#         "input_edits": len(mquake_clean),
#         "input_windows": sum(len(row["variants"][0]["portability"]) for row in mquake_clean),
#         "multi_window_edits": multi_window_edits,
#         "rejected_windows": dict(rejected_windows),
#         "dropped_edits": dropped_edits,
#         "direct_prompt_conflict_repairs": conflict_repairs,
#         "remaining_direct_prompt_conflicts": 0,
#         "uniform_records": len(mquake_uniform),
#         "selected_source_hop_counts": dict(hop_counts),
#         "selected_alignment": dict(alignment_counts),
#         "selected_directions": dict(direction_counts),
#         "selected_source_configs": dict(source_config_counts),
#         "quality_ties_resolved_by_hash": quality_ties,
#         "rewrite_required_records": sum(row["portability"]["rewrite_required"] for row in mquake_uniform),
#         "direct_semantic_rewrite_records": sum(
#             row["portability"]["rewrite_required"]
#             and "source_chain_length" not in row["portability"]["rewrite_reasons"]
#             for row in mquake_uniform
#         ),
#         "rewrite_reasons": dict(rewrite_reason_counts),
#     }, file, ensure_ascii=False, indent=2)

In [ ]:
# from datasets import load_from_disk


# mquake_uniform = load_from_disk("../data/datasets/mquake/uniform")
# mquake_uniform

### Prompt Rewriting

In [ ]:
# import asyncio
# import json
# import re
# from collections import Counter
# from copy import deepcopy
# from difflib import SequenceMatcher

# from datasets import Dataset
# from openai import AsyncOpenAI
# from tqdm.auto import tqdm


# EXPECTED_PORTABILITY = 3
# REWRITE_GUIDANCE = {
#     "strict_prompt_validation": (
#         "The original prompts failed deterministic semantic validation. Follow the canonical prompt and the exact two-hop "
#         "relations; do not preserve questionable wording from the source."
#     ),
#     "official_language": (
#         "P37 means official language. Every question must explicitly say official language; generic language spoken or "
#         "used in the country is not precise enough."
#     ),
#     "performer_direction": (
#         "P175 maps the chain-start work or entity to its performer. Ask for the second-hop property of the performer of "
#         "the chain-start subject; never describe the chain-start subject as a work performed by itself."
#     ),
#     "citizenship_scope": (
#         "P27 means country of citizenship. Preserve citizenship explicitly; do not replace it with country of origin, "
#         "where the person is from, birthplace, or generic nationality wording."
#     ),
#     "notable_work": (
#         "P800 means notable work. Ask for a notable work, not a claim to fame, achievement, distinction, or anything else "
#         "the intermediate entity may be known for."
#     ),
#     "work_location": (
#         "P937 means work location. Explicitly ask where the intermediate person works or worked, or for their workplace; "
#         "do not ask for a generic associated location."
#     ),
#     "ceo_relation": (
#         "P169 means chief executive officer. Ask for the CEO or chief executive officer, not a generic head, leader, "
#         "manager, or chairperson."
#     ),
#     "chairperson_relation": (
#         "P488 means chairperson. Ask for the chairperson or person who chairs the intermediate entity, not a generic head "
#         "or leader."
#     ),
#     "manager_relation": (
#         "P1037 means director or manager. Ask who directs or manages the intermediate entity; generic head or person in "
#         "charge wording is not precise enough."
#     ),
#     "head_of_state_relation": (
#         "P35 means head of state. Explicitly preserve head-of-state semantics; do not reduce it to a generic current "
#         "leader question."
#     ),
#     "head_of_government_relation": (
#         "P6 means head of government. Explicitly preserve head-of-government semantics; do not reduce it to a generic "
#         "current leader question."
#     ),
#     "source_chain_length": (
#         "The original source question described a longer chain. Ignore its wording and ask only for the direct "
#         "answer of the selected two-hop chain."
#     ),
#     "generic_genre": (
#         "P136 means general genre. Do not assume that the intermediate entity creates, performs, or plays music. "
#         "Ask for the genre associated with the intermediate entity; valid answers may be literary, theatrical, "
#         "cinematic, musical, or another kind of genre."
#     ),
#     "performer_manager": (
#         "P1037 identifies the manager or director associated with the performer. Ask who manages or directs the "
#         "performer, not who directed a performance, recording, film, or song."
#     ),
#     "birthplace_wording": (
#         "P19 means place of birth. Ask where the intermediate entity was born. Hometown, origin, residence, and "
#         "nationality are not equivalent to birthplace."
#     ),
#     "language_context": (
#         "P407 means language of work or name. Ask for the natural language associated with the intermediate work, "
#         "name, or organization. Do not reinterpret it as a software programming language or say that a company "
#         "itself was written."
#     ),
#     "original_language": (
#         "P364 means the original language of a film or television work. Ask for its original language, not its "
#         "native language and not the language of a person."
#     ),
#     "officeholder_reference": (
#         "P1308 returns the person who holds the named office or position. The second relation applies to that "
#         "person, not to the office, title, or institution itself."
#     ),
#     "location_type_assumption": (
#         "Ask for the location without assuming that it must be a city. The answer may be a city, country, state, "
#         "region, building, or another location type."
#     ),
#     "founder_type_assumption": (
#         "P112 asks which person or organization founded the intermediate entity. Do not assume that the founder "
#         "must be a human."
#     ),
#     "founder_scope": (
#         "The second hop is P112 founded by. Ask who or what founded the intermediate country or entity. The answer "
#         "must be its founder, never the country or entity itself."
#     ),
#     "child_direction": (
#         "The second hop is P40 child. Ask for the child of the intermediate person. Do not reverse the question into "
#         "asking whose child that person is."
#     ),
#     "spoken_language_scope": (
#         "P1412 means languages spoken, written, or signed. Do not narrow it to native language, mother tongue, or "
#         "first language."
#     ),
#     "headquarters_relation": (
#         "P159 specifically means headquarters location. Every question must explicitly preserve headquarters, head "
#         "office, or organizational base semantics rather than asking for a generic location."
#     ),
#     "formation_location_relation": (
#         "P740 specifically means location of formation. Every question must ask where the intermediate entity was "
#         "formed, founded, established, or originated rather than asking for a generic associated location."
#     ),
#     "genre_context": (
#         "The first hop uses P136 genre. Refer to it as a genre without assuming it is music or something that is "
#         "played or performed."
#     ),
#     "intermediate_location_type": (
#         "The first hop returns a location. Refer to it as a place or location without assuming it is a city, town, "
#         "or municipality."
#     ),
#     "relation_scope": (
#         "Apply the second relation to the intermediate entity returned by hop 1, not to the chain-start entity or an "
#         "entity associated with the intermediate one."
#     ),
#     "possessive_scope": (
#         "Preserve the hop order in possessive phrasing. Ask for the second-hop property of the intermediate entity, "
#         "not for the first-hop relation of an entity named by the second hop."
#     ),
# }


# def normalize_prompt(text: str) -> str:
#     return " ".join(str(text or "").strip().split())


# def prompt_key(text: str) -> str:
#     text = re.sub(r"[^0-9a-z]+", " ", str(text or "").casefold())
#     return " ".join(text.split())


# def contains_phrase(prompt: str, phrase: str, min_len: int = 1) -> bool:
#     phrase_key = prompt_key(phrase)
#     if len(phrase_key) < min_len:
#         return False
#     return f" {phrase_key} " in f" {prompt_key(prompt)} "


# def remove_prompt_phrase(prompt: str, phrase: str) -> str:
#     prompt_marker = f" {prompt_key(prompt)} "
#     phrase_marker = f" {prompt_key(phrase)} "
#     return " ".join(prompt_marker.replace(phrase_marker, " ").split())


# def answer_leak(prompt: str, answers: list[str]) -> str | None:
#     for answer in answers:
#         if contains_phrase(prompt, answer, min_len=4):
#             return answer
#     return None


# def strip_json_response(text: str) -> str:
#     text = str(text or "").strip()
#     if text.startswith("```"):
#         text = re.sub(r"^```(?:json)?\s*", "", text)
#         text = re.sub(r"\s*```$", "", text)
#     return text.strip()


# def add_reason(reasons: list[str], reason: str) -> None:
#     if reason not in reasons:
#         reasons.append(reason)


# def detect_rewrite_reasons(record: dict) -> list[str]:
#     portability = record["portability"]
#     hop1, hop2 = portability["hops"]["new"]
#     relation1 = hop1["relation"]["id"]
#     relation2 = hop2["relation"]["id"]
#     prompts = [prompt.casefold() for prompt in portability.get("prompts", [])]
#     reasons = list(portability.get("rewrite_reasons", []))

#     if relation1 == "P136" and any(
#         any(cue in prompt for cue in ("music", "played", "plays", "performed"))
#         for prompt in prompts
#     ):
#         add_reason(reasons, "genre_context")
#     if (relation1, relation2) == ("P27", "P112"):
#         add_reason(reasons, "founder_scope")
#     if relation2 == "P40":
#         add_reason(reasons, "child_direction")
#     if relation2 == "P1412" and any(
#         "native language" in prompt or "native tongue" in prompt
#         for prompt in prompts
#     ):
#         add_reason(reasons, "spoken_language_scope")
#     if relation2 == "P159" and any(
#         "city" in prompt
#         or not any(cue in prompt for cue in ("headquarter", "head office", "main office", "based"))
#         for prompt in prompts
#     ):
#         add_reason(reasons, "headquarters_relation")
#     if relation2 == "P740" and any(
#         "city" in prompt
#         or not any(cue in prompt for cue in ("formation", "formed", "founded", "origin", "established"))
#         for prompt in prompts
#     ):
#         add_reason(reasons, "formation_location_relation")
#     if relation1 in {"P19", "P20", "P159", "P740", "P937"} and any(
#         any(place_type in prompt.split() for place_type in ("city", "town", "municipality"))
#         for prompt in prompts
#     ):
#         add_reason(reasons, "intermediate_location_type")
#     if (relation1, relation2) == ("P35", "P170"):
#         add_reason(reasons, "relation_scope")
#     if relation2 == "P26":
#         add_reason(reasons, "possessive_scope")
#     return reasons


# def prepare_record(record: dict) -> dict:
#     record = deepcopy(dict(record))
#     record["portability"] = deepcopy(dict(record["portability"]))
#     reasons = detect_rewrite_reasons(record)
#     record["portability"]["rewrite_required"] = bool(reasons)
#     record["portability"]["rewrite_reasons"] = reasons

#     return record


# def build_intermediate_phrase(record: dict) -> str:
#     hop1 = record["portability"]["hops"]["new"][0]
#     start_subject = hop1["subject"]["str"]
#     relation_id = hop1["relation"]["id"]
#     if relation_id == "P1308":
#         return f"the person who holds the office of {start_subject}"
#     if relation_id == "P112":
#         return f"the founder of {start_subject}"
#     return f"the {hop1['relation']['str']} of {start_subject}"


# def build_canonical_prompt(record: dict) -> str:
#     hop2 = record["portability"]["hops"]["new"][1]
#     intermediate = build_intermediate_phrase(record)
#     relation2 = hop2["relation"]["id"]
#     if relation2 == "P19":
#         return f"Where was {intermediate} born?"
#     if relation2 == "P136":
#         return f"What genre is associated with {intermediate}?"
#     if relation2 == "P1037":
#         return f"Who is the manager or director associated with {intermediate}?"
#     if relation2 == "P364":
#         return f"What was the original language of {intermediate}?"
#     if relation2 == "P1412":
#         return f"What languages does {intermediate} speak, write, or sign?"
#     if relation2 == "P159":
#         return f"What is the headquarters location of {intermediate}?"
#     if relation2 == "P740":
#         return f"Where was {intermediate} formed or established?"
#     if relation2 == "P169":
#         return f"Who is the chief executive officer of {intermediate}?"
#     if relation2 == "P488":
#         return f"Who is the chairperson of {intermediate}?"
#     if relation2 == "P35":
#         return f"Who is the head of state of {intermediate}?"
#     if relation2 == "P6":
#         return f"Who is the head of government of {intermediate}?"
#     if relation2 == "P37":
#         return f"What is the official language of {intermediate}?"
#     if relation2 == "P800":
#         return f"What notable work is associated with {intermediate}?"
#     if relation2 == "P937":
#         return f"Where does {intermediate} work?"
#     if relation2 == "P20":
#         return f"Where did {intermediate} die?"
#     if relation2 == "P112":
#         return f"Who or what founded {intermediate}?"
#     if relation2 == "P40":
#         return f"Who is a child of {intermediate}?"
#     if relation2 == "P170":
#         return f"Who or what created {intermediate}?"
#     if relation2 == "P26":
#         return f"Who is the spouse of {intermediate}?"
#     return f"What is the {hop2['relation']['str']} of {intermediate}?"


# def build_fallback_prompts(record: dict) -> list[str]:
#     intermediate = build_intermediate_phrase(record)
#     relation = record["portability"]["hops"]["new"][1]["relation"]
#     relation_id = relation["id"]
#     prompts_by_relation = {
#         "P6": [
#             f"Who is the head of government of {intermediate}?",
#             f"Which person serves as head of government for {intermediate}?",
#             f"Who holds the head-of-government position in {intermediate}?",
#         ],
#         "P19": [
#             f"Where was {intermediate} born?",
#             f"What is the birthplace of {intermediate}?",
#             f"At what location did the birth of {intermediate} occur?",
#         ],
#         "P20": [
#             f"Where did {intermediate} die?",
#             f"What was the place of death of {intermediate}?",
#             f"At what location did {intermediate} pass away?",
#         ],
#         "P26": [
#             f"Who is the spouse of {intermediate}?",
#             f"Which person is married to {intermediate}?",
#             f"What person is the marital partner of {intermediate}?",
#         ],
#         "P27": [
#             f"What is the country of citizenship of {intermediate}?",
#             f"Which country is {intermediate} a citizen of?",
#             f"What country grants citizenship to {intermediate}?",
#         ],
#         "P35": [
#             f"Who is the head of state of {intermediate}?",
#             f"Which person serves as head of state for {intermediate}?",
#             f"Who holds the head-of-state position in {intermediate}?",
#         ],
#         "P37": [
#             f"What is the official language of {intermediate}?",
#             f"Which language has official status in {intermediate}?",
#             f"What language is officially recognized by {intermediate}?",
#         ],
#         "P40": [
#             f"Who is a child of {intermediate}?",
#             f"Which child belongs to {intermediate}?",
#             f"What child does {intermediate} have?",
#         ],
#         "P136": [
#             f"What genre is associated with {intermediate}?",
#             f"Which genre does {intermediate} belong to?",
#             f"How is the genre of {intermediate} classified?",
#         ],
#         "P1412": [
#             f"What languages does {intermediate} speak, write, or sign?",
#             f"Which languages are spoken, written, or signed by {intermediate}?",
#             f"What language or languages can {intermediate} use in speech, writing, or signing?",
#         ],
#         "P159": [
#             f"What is the headquarters location of {intermediate}?",
#             f"Where is the head office of {intermediate}?",
#             f"What location serves as the organizational base of {intermediate}?",
#         ],
#         "P169": [
#             f"Who is the chief executive officer of {intermediate}?",
#             f"Which person serves as CEO of {intermediate}?",
#             f"Who holds the chief-executive role at {intermediate}?",
#         ],
#         "P488": [
#             f"Who is the chairperson of {intermediate}?",
#             f"Which person chairs {intermediate}?",
#             f"Who holds the chairperson role at {intermediate}?",
#         ],
#         "P740": [
#             f"Where was {intermediate} formed?",
#             f"At what location was {intermediate} established?",
#             f"Where did the formation of {intermediate} occur?",
#         ],
#         "P800": [
#             f"What notable work is associated with {intermediate}?",
#             f"Which notable work was produced by {intermediate}?",
#             f"What is a notable work of {intermediate}?",
#         ],
#         "P937": [
#             f"Where does {intermediate} work?",
#             f"What is the work location of {intermediate}?",
#             f"Which workplace is associated with {intermediate}?",
#         ],
#         "P1037": [
#             f"Who manages or directs {intermediate}?",
#             f"Which manager or director is associated with {intermediate}?",
#             f"Who is responsible for managing or directing {intermediate}?",
#         ],
#         "P140": [
#             f"What is the religion or worldview of {intermediate}?",
#             f"Which religion or worldview is associated with {intermediate}?",
#             f"What religious or philosophical worldview does {intermediate} follow?",
#         ],
#         "P495": [
#             f"What is the country of origin of {intermediate}?",
#             f"Which country does {intermediate} originate from?",
#             f"What country is associated with the origin of {intermediate}?",
#         ],
#         "P106": [
#             f"What is the occupation of {intermediate}?",
#             f"What profession does {intermediate} have?",
#             f"Which occupation is associated with {intermediate}?",
#         ],
#         "P170": [
#             f"Who or what created {intermediate}?",
#             f"Which creator is responsible for {intermediate}?",
#             f"What person or organization created {intermediate}?",
#         ],
#         "P176": [
#             f"Who or what manufactured {intermediate}?",
#             f"Which manufacturer produced {intermediate}?",
#             f"What organization manufactured {intermediate}?",
#         ],
#         "P407": [
#             f"What is the language of {intermediate}?",
#             f"Which language is associated with {intermediate}?",
#             f"In what language is {intermediate} presented?",
#         ],
#         "P364": [
#             f"What was the original language of {intermediate}?",
#             f"In which language was {intermediate} originally produced?",
#             f"What language was originally used for {intermediate}?",
#         ],
#         "P449": [
#             f"What was the original broadcaster of {intermediate}?",
#             f"Which network or service originally broadcast {intermediate}?",
#             f"Who or what first broadcast {intermediate}?",
#         ],
#     }
#     if relation_id in prompts_by_relation:
#         return prompts_by_relation[relation_id]
#     relation_name = relation["str"]
#     return [
#         f"What is the {relation_name} of {intermediate}?",
#         f"Which answer gives the {relation_name} associated with {intermediate}?",
#         f"Regarding {intermediate}, what is its {relation_name}?",
#     ]


# def compact_record(record: dict, include_prompts: bool = True) -> dict:
#     portability = record["portability"]
#     compact = {
#         "canonical_prompt": build_canonical_prompt(record),
#         "rewrite_reasons": list(portability.get("rewrite_reasons", [])),
#         "expected_answers": list(portability["answers"]["new"]),
#         "hops": [
#             {
#                 "subject": hop["subject"]["str"],
#                 "relation_id": hop["relation"]["id"],
#                 "relation": hop["relation"]["str"],
#                 "answer": hop["answers"]["str"][0],
#             }
#             for hop in portability["hops"]["new"]
#         ],
#     }
#     if include_prompts:
#         compact["prompts"] = list(portability.get("prompts", []))
#     return compact


# def finalize_record(record: dict) -> dict:
#     record = deepcopy(dict(record))
#     record["portability"] = deepcopy(dict(record["portability"]))
#     record["portability"].pop("rewrite_required", None)
#     record["portability"].pop("rewrite_reasons", None)
#     return record


# def validate_semantic_guidance(record: dict, prompt: str, prompt_without_start: str) -> list[str]:
#     errors = []
#     lowered = prompt_without_start.casefold()
#     original_lowered = prompt.casefold()
#     portability = record["portability"]
#     hop1, hop2 = portability["hops"]["new"]
#     relation1 = hop1["relation"]["id"]
#     relation2 = hop2["relation"]["id"]
#     reasons = set(portability.get("rewrite_reasons", []))

#     if "generic_genre" in reasons and any(word in lowered.split() for word in ("music", "musical", "play", "plays")):
#         errors.append("assumes P136 is a music genre")
#     if "performer_manager" in reasons and "performance" in lowered:
#         errors.append("asks who directed a performance instead of who manages the performer")
#     if "birthplace_wording" in reasons and any(word in lowered for word in ("hometown", "origin", "originate")):
#         errors.append("uses hometown or origin instead of birthplace")
#     if "language_context" in reasons and "programming language" in lowered:
#         errors.append("confuses P407 with a programming language")
#     if "original_language" in reasons and "native language" in lowered:
#         errors.append("uses native language instead of original language")
#     if "officeholder_reference" in reasons:
#         holder_cues = (
#             "officeholder", "holds the office", "holding the office", "holds the position", "holding the position",
#             "holder of", "serves as", "serving as", "person in the role", "current", "incumbent",
#         )
#         if "entity known as" in lowered or not any(cue in lowered for cue in holder_cues):
#             errors.append("does not clearly refer to the person holding the office")
#     if "location_type_assumption" in reasons and "city" in lowered.split():
#         errors.append("assumes the final location must be a city")
#     if "founder_type_assumption" in reasons and any(
#         phrase in lowered for phrase in ("person who founded", "name of the person", "which person founded")
#     ):
#         errors.append("assumes the founder must be a person")
#     if "founder_scope" in reasons:
#         if re.match(r"^(what|which) country\b", lowered):
#             errors.append("asks for the intermediate country instead of its founder")
#         if any(phrase in lowered for phrase in ("has a founder", "known founder", "notable founder")):
#             errors.append("describes a founder without asking for the founder")
#         if not any(cue in lowered for cue in ("founder", "founded", "established", "establish")):
#             errors.append("does not preserve the founded-by relation")
#     if "child_direction" in reasons:
#         if "whose child" in lowered or re.search(r"whose .+ is the (?:child|offspring)", lowered):
#             errors.append("reverses the child relation")
#         if not any(cue in lowered for cue in ("child", "offspring", "son", "daughter")):
#             errors.append("does not ask for the intermediate person's child")
#     if "spoken_language_scope" in reasons:
#         if any(
#             cue in lowered for cue in ("native language", "native tongue", "mother tongue", "first language")
#         ):
#             errors.append("narrows P1412 to native language")
#         if "language" not in lowered and "tongue" not in lowered:
#             errors.append("does not ask for a spoken, written, or signed language")
#     if "headquarters_relation" in reasons and not any(
#         cue in lowered for cue in ("headquarter", "head office", "main office", "organizational base", "based")
#     ):
#         errors.append("drops the headquarters-location relation")
#     if "formation_location_relation" in reasons and not any(
#         cue in lowered for cue in ("formation", "formed", "founded", "origin", "established")
#     ):
#         errors.append("drops the location-of-formation relation")
#     if "genre_context" in reasons and any(
#         cue in lowered for cue in ("music", "played", "plays", "performed")
#     ):
#         errors.append("assumes the first-hop genre is music")
#     if "intermediate_location_type" in reasons and any(
#         place_type in lowered.split() for place_type in ("city", "town", "municipality")
#     ):
#         errors.append("assumes the intermediate location has a specific place type")
#     if "relation_scope" in reasons and any(
#         phrase in lowered for phrase in ("entity led by", "entity that is currently headed", "same head of state")
#     ):
#         errors.append("applies the second relation to the wrong entity")
#     if "possessive_scope" in reasons and re.search(
#         r"\bof .+['’]s spouse\b", original_lowered
#     ):
#         errors.append("reverses the possessive hop order")

#     if relation1 == "P175":
#         wrong_direction = any(
#             cue in original_lowered
#             for cue in ("work performed by", "song performed by", "film performed by")
#         )
#         performer_cues = ("perform", "portray", "actor", "singer", "artist", "act ")
#         if wrong_direction:
#             errors.append("reverses the P175 performer direction")
#         elif not any(cue in original_lowered for cue in performer_cues):
#             errors.append("does not preserve the P175 performer relation")
#     if relation1 == "P27":
#         if (
#             "country of origin" in original_lowered
#             or re.search(r"\bwhere .+ is from\b", original_lowered)
#             or " comes from" in original_lowered
#         ):
#             errors.append("replaces citizenship with country of origin")
#         if not any(cue in lowered for cue in ("citizen", "citizenship")):
#             errors.append("does not preserve the P27 citizenship relation")
#     if relation1 == "P800" and "notable work" not in lowered:
#         errors.append("does not preserve the first-hop P800 notable-work relation")
#     if relation1 == "P35" and "head of state" not in lowered:
#         errors.append("does not preserve the first-hop P35 head-of-state relation")
#     if relation1 == "P169" and not any(cue in lowered for cue in ("chief executive", "ceo")):
#         errors.append("does not preserve the first-hop P169 chief-executive relation")
#     if relation1 == "P1037" and not any(
#         cue in lowered for cue in ("manager", "manages", "managing", "director", "directs", "directing")
#     ):
#         errors.append("does not preserve the first-hop P1037 manager/director relation")
#     if relation1 == "P26" and not any(cue in lowered for cue in ("spouse", "married", "marital")):
#         errors.append("uses a broader partner relation instead of first-hop P26 spouse")
#     if relation2 == "P37" and "official" not in lowered:
#         errors.append("drops the official-language relation")
#     if relation2 == "P800" and "notable work" not in lowered:
#         errors.append("asks for a broad claim to fame instead of a notable work")
#     if relation2 == "P937" and not any(
#         cue in lowered for cue in ("work", "workplace", "employ", "job")
#     ):
#         errors.append("drops the work-location relation")
#     if relation2 == "P169" and not any(cue in lowered for cue in ("chief executive", "ceo")):
#         errors.append("does not preserve the P169 chief-executive relation")
#     if relation2 == "P488" and not any(
#         cue in lowered for cue in ("chairperson", "chairman", "chairwoman", "chairs")
#     ):
#         errors.append("does not preserve the P488 chairperson relation")
#     if relation2 == "P1037" and not any(
#         cue in lowered for cue in ("manager", "manages", "managing", "director", "directs", "directing")
#     ):
#         errors.append("does not preserve the P1037 manager/director relation")
#     if relation2 == "P35" and "head of state" not in lowered:
#         errors.append("does not preserve the P35 head-of-state relation")
#     if relation2 == "P6" and not any(
#         cue in lowered for cue in ("head of government", "head of the government", "prime minister", "premier", "leader of government", "leader of the government", "heads the government", "leads the government")
#     ):
#         errors.append("does not preserve the P6 head-of-government relation")
#     if relation2 == "P1412":
#         if any(
#             cue in lowered for cue in ("native language", "native tongue", "mother tongue", "first language")
#         ):
#             errors.append("narrows P1412 to native language")
#         if "language" not in lowered and "tongue" not in lowered:
#             errors.append("does not ask for a spoken, written, or signed language")
#     if relation2 == "P159" and not any(
#         cue in lowered for cue in ("headquarter", "head office", "main office", "organizational base", "based")
#     ):
#         errors.append("drops the headquarters-location relation")
#     if relation2 == "P740" and not any(
#         cue in lowered for cue in ("formation", "formed", "founded", "origin", "established")
#     ):
#         errors.append("drops the location-of-formation relation")
#     if relation2 == "P40" and re.search(
#         r"\bof .+['’]s (?:child|offspring)\b", original_lowered
#     ):
#         errors.append("reverses the child possessive scope")
#     if relation2 == "P26" and not any(cue in lowered for cue in ("spouse", "married", "marital")):
#         errors.append("uses a broader partner relation instead of P26 spouse")
#     if relation2 == "P27" and not any(cue in lowered for cue in ("citizen", "citizenship")):
#         errors.append("uses nationality instead of the P27 citizenship relation")
#     if relation2 == "P170" and not any(
#         cue in lowered for cue in ("create", "created", "creator", "author", "wrote", "written", "made")
#     ):
#         errors.append("does not preserve the P170 creator relation")
#     if "the founded by of" in original_lowered:
#         errors.append("contains an ungrammatical founded-by relation phrase")
#     if relation2 in {"P140", "P495"} and (
#         "which answer gives" in original_lowered or original_lowered.startswith("regarding ")
#     ):
#         errors.append("uses generic fallback scaffolding")
#     return list(dict.fromkeys(errors))

# def prompt_pair_similarity(prompt1: str, prompt2: str, chain_start: str) -> float:
#     chain_start_key = prompt_key(chain_start)
#     left = prompt_key(prompt1).replace(chain_start_key, " ")
#     right = prompt_key(prompt2).replace(chain_start_key, " ")
#     return SequenceMatcher(None, left, right).ratio()


# def validate_rewritten_prompts(record: dict, prompts: list[str]) -> list[str]:
#     errors = []
#     prompts = [normalize_prompt(prompt) for prompt in prompts]

#     if len(prompts) != EXPECTED_PORTABILITY:
#         return [f"expected {EXPECTED_PORTABILITY} prompts, got {len(prompts)}"]

#     keys = [prompt_key(prompt) for prompt in prompts]
#     if len(set(keys)) != len(keys):
#         errors.append("duplicate prompts")

#     portability = record["portability"]
#     expected_answers = portability["answers"]["new"]
#     chain_start = portability["hops"]["new"][0]["subject"]["str"]
#     intermediate = portability["hops"]["new"][0]["answers"]["str"][0]
#     chain_start_key = prompt_key(chain_start)
#     for left_index in range(len(prompts)):
#         for right_index in range(left_index + 1, len(prompts)):
#             similarity = prompt_pair_similarity(prompts[left_index], prompts[right_index], chain_start)
#             if similarity >= 0.95:
#                 errors.append(
#                     f"prompts {left_index} and {right_index} are too lexically similar ({similarity:.3f})"
#                 )

#     for index, prompt in enumerate(prompts):
#         lowered = prompt.casefold()
#         if not prompt:
#             errors.append(f"prompt {index}: empty")
#             continue
#         if not prompt.endswith("?"):
#             errors.append(f"prompt {index}: not a question")
#         if re.match(r"^\s*(\d+[).:-]|[-*])\s+", prompt):
#             errors.append(f"prompt {index}: contains bullet or numbering")
#         if any(token in lowered for token in ("```", "prompt:", "answer:", "record ", "question 1")):
#             errors.append(f"prompt {index}: contains labels or markdown")
#         if chain_start_key and chain_start_key not in prompt_key(prompt):
#             errors.append(f"prompt {index}: missing chain-start subject `{chain_start}`")

#         prompt_without_start = remove_prompt_phrase(prompt, chain_start)
#         if contains_phrase(prompt_without_start, intermediate, min_len=4):
#             errors.append(f"prompt {index}: directly names intermediate entity `{intermediate}`")

#         leaked_answer = answer_leak(prompt_without_start, expected_answers)
#         if leaked_answer:
#             errors.append(f"prompt {index}: leaks expected answer `{leaked_answer}`")
#         for semantic_error in validate_semantic_guidance(record, prompt, prompt_without_start):
#             errors.append(f"prompt {index}: {semantic_error}")
#     return errors


# def select_examples(dataset, max_examples: int = 4) -> list[dict]:
#     examples = []
#     for record in dataset:
#         if record["portability"].get("rewrite_required"):
#             continue
#         errors = validate_rewritten_prompts(record, record["portability"].get("prompts", []))
#         if errors:
#             continue
#         examples.append(compact_record(record, include_prompts=True))
#         if len(examples) >= max_examples:
#             break
#     return examples


# def build_rewrite_prompt(record: dict, examples: list[dict], feedback: dict | None = None) -> str:
#     example_payload = json.dumps(examples, ensure_ascii=False, indent=2)
#     task_payload = json.dumps(compact_record(record, include_prompts=False), ensure_ascii=False, indent=2)
#     guidance = [
#         REWRITE_GUIDANCE[reason]
#         for reason in record["portability"].get("rewrite_reasons", [])
#         if reason in REWRITE_GUIDANCE
#     ]
#     guidance_text = "\n".join(f"- {item}" for item in guidance) or "- Follow the exact two-hop chain without adding assumptions."
#     feedback_text = ""
#     if feedback:
#         feedback_text = (
#             "\n\nThe previous attempt was rejected. Correct every listed problem:\n"
#             f"{json.dumps(feedback, ensure_ascii=False, indent=2)}"
#         )
#     return (
#         "Paraphrase one canonical MQuAKE portability question for a model-editing benchmark.\n"
#         "Only write prompt strings. Hop subjects, relations, answers, and list lengths are locked.\n"
#         "The input has a two-hop chain and a canonical_prompt. The canonical_prompt and hop data are the source of truth.\n"
#         "The original prompts are intentionally omitted because their chain length or semantics may be wrong.\n"
#         "Return one JSON object containing only a prompts list.\n\n"
#         "Record-specific semantic guidance:\n"
#         f"{guidance_text}\n\n"
#         "Hard semantic rule:\n"
#         "The expected answer is the direct answer to hop 2. Do not ask for a property of that answer and do not expose the intermediate entity.\n\n"
#         "Rules:\n"
#         "1) Return exactly 3 natural questions.\n"
#         "2) Each question must preserve the exact meaning of canonical_prompt and the two hop relations.\n"
#         "3) Each question must require following hop 1 from the chain-start subject, then hop 2.\n"
#         "4) Include the chain-start subject exactly as written.\n"
#         "5) Refer to the intermediate entity only through its relation to the chain-start subject.\n"
#         "6) Do not include, paraphrase, hint, or leak any expected answer.\n"
#         "7) Keep every prompt as a question ending with a question mark.\n"
#         "8) Do not add labels, numbering, bullets, explanations, markdown, or extra text inside prompts.\n"
#         "9) The 3 questions must use meaningfully different wording.\n\n"
#         "Valid examples are provided only for style, never for content:\n"
#         f"{example_payload}\n\n"
#         "Input record:\n"
#         f"{task_payload}"
#         f"{feedback_text}\n\n"
#         "Output JSON only. Format:\n"
#         "{\"prompts\": [\"question 1?\", \"question 2?\", \"question 3?\"]}"
#     )


# def parse_rewrite_response(content: str) -> list[str]:
#     content = strip_json_response(content)
#     try:
#         payload = json.loads(content)
#     except Exception:
#         start = content.find("{")
#         end = content.rfind("}")
#         if start == -1 or end == -1 or end < start:
#             raise ValueError("response does not contain a JSON object")
#         payload = json.loads(content[start:end + 1])

#     if not isinstance(payload, dict) or not isinstance(payload.get("prompts"), list):
#         raise ValueError("response must be a JSON object with a prompts list")
#     return [normalize_prompt(prompt) for prompt in payload["prompts"]]


# async def rewrite_record(
#     record: dict,
#     examples: list[dict],
#     client: AsyncOpenAI,
#     config: dict,
#     feedback: dict | None = None,
# ) -> tuple[list[str] | None, list[str], dict | None]:
#     messages = [
#         {
#             "role": "system",
#             "content": "You paraphrase benchmark questions. You only change prompt text and return valid JSON.",
#         },
#         {
#             "role": "user",
#             "content": build_rewrite_prompt(record, examples, feedback),
#         },
#     ]
#     completion = await client.chat.completions.create(
#         model=config["model"],
#         messages=messages,
#         **config.get("request_config", {}),
#     )
#     content = completion.choices[0].message.content
#     if content is None:
#         return None, ["empty model response"], None

#     try:
#         prompts = parse_rewrite_response(content)
#     except Exception as exc:
#         return None, [f"invalid JSON: {exc}"], {"raw_response": strip_json_response(content)}

#     errors = validate_rewritten_prompts(record, prompts)
#     if errors:
#         return None, errors, {"prompts": prompts}
#     return prompts, [], None


# async def rewrite_dataset(
#     dataset,
#     config: dict,
#     start: int = 0,
#     limit: int | None = None,
# ) -> tuple[Dataset, list[dict], dict]:
#     if not config.get("model"):
#         raise ValueError("config['model'] must be specified")
#     if start < 0:
#         raise ValueError("start must be non-negative")
#     if limit is not None and limit < 0:
#         raise ValueError("limit must be non-negative when set")

#     end = len(dataset) if limit is None else min(len(dataset), start + limit)
#     source_indices = list(range(start, end))
#     rows = [prepare_record(dataset[index]) for index in source_indices]
#     examples = select_examples(rows, max_examples=4)
#     tasks = [
#         (slot, source_index, record)
#         for slot, (source_index, record) in enumerate(zip(source_indices, rows))
#         if record["portability"].get("rewrite_required")
#     ]

#     client = AsyncOpenAI(base_url=config.get("base_url"), api_key=config.get("api_key"))
#     semaphore = asyncio.Semaphore(config.get("max_concurrency", 4))
#     max_retries = config.get("max_retries", 3)
#     retry_delay = config.get("retry_delay", 2.0)
#     failures = []
#     rewritten_indices = set()
#     fallback_indices = set()

#     async def worker(slot: int, source_index: int, record: dict) -> None:
#         last_errors = []
#         last_rejected_output = None
#         for attempt in range(max_retries + 1):
#             feedback = None
#             if last_errors:
#                 feedback = {"errors": last_errors, "rejected_output": last_rejected_output}
#             try:
#                 async with semaphore:
#                     prompts, errors, rejected_output = await rewrite_record(
#                         record,
#                         examples,
#                         client,
#                         config,
#                         feedback,
#                     )
#             except Exception as exc:
#                 prompts, errors, rejected_output = None, [repr(exc)], None

#             if prompts is not None:
#                 rows[slot]["portability"]["prompts"] = prompts
#                 rewritten_indices.add(source_index)
#                 return

#             last_errors = errors
#             last_rejected_output = rejected_output
#             if attempt < max_retries:
#                 await asyncio.sleep(retry_delay * (attempt + 1))

#         fallback_prompts = build_fallback_prompts(record)
#         fallback_errors = validate_rewritten_prompts(record, fallback_prompts)
#         if not fallback_errors:
#             rows[slot]["portability"]["prompts"] = fallback_prompts
#             fallback_indices.add(source_index)
#             return

#         failures.append({
#             "index": source_index,
#             "errors": [*last_errors, *fallback_errors],
#             "record": compact_record(record, include_prompts=False),
#             "rewritten": last_rejected_output,
#         })

#     running_tasks = [asyncio.create_task(worker(*task)) for task in tasks]
#     try:
#         for task in tqdm(asyncio.as_completed(running_tasks), total=len(running_tasks), desc="Rewriting MQuAKE"):
#             await task
#     finally:
#         for task in running_tasks:
#             if not task.done():
#                 task.cancel()
#         await asyncio.gather(*running_tasks, return_exceptions=True)
#         await client.close()

#     # Repair non-task rows deterministically so stale source wording cannot leak into final.
#     for slot, (source_index, record) in enumerate(zip(source_indices, rows)):
#         errors = validate_rewritten_prompts(record, record["portability"].get("prompts", []))
#         if not errors:
#             continue
#         fallback_prompts = build_fallback_prompts(record)
#         fallback_errors = validate_rewritten_prompts(record, fallback_prompts)
#         if fallback_errors:
#             failures.append({
#                 "index": source_index,
#                 "errors": fallback_errors,
#                 "record": compact_record(record, include_prompts=True),
#                 "rewritten": {"prompts": record["portability"].get("prompts", [])},
#             })
#             continue
#         rows[slot]["portability"]["prompts"] = fallback_prompts
#         fallback_indices.add(source_index)

#     reason_counts = Counter(
#         reason
#         for _, _, record in tasks
#         for reason in record["portability"].get("rewrite_reasons", [])
#     )
#     report = {
#         "dataset_records": len(dataset),
#         "selected_start": start,
#         "selected_end": end,
#         "selected_records": len(rows),
#         "rewrite_required_records": len(tasks),
#         "rewrite_reasons": dict(reason_counts),
#         "rewritten_records": len(rewritten_indices),
#         "fallback_records": len(fallback_indices),
#         "candidate_records": len(rows),
#         "failed_records": len(failures),
#     }
#     return Dataset.from_list(rows), failures, report

In [ ]:
# import os
# from dotenv import load_dotenv


# load_dotenv()

# rewrite_config = {
#     "base_url": "http://localhost:20128/v1",
#     "api_key": os.environ["9ROUTER_API_KEY"],
#     "model": "cx/gpt-5.6-luna(low)",
#     "max_concurrency": 50,
#     "max_retries": 5,
#     "retry_delay": 2.0,
# }

# mquake_rewritten, rewrite_failures, rewrite_report = await rewrite_dataset(
#     mquake_uniform,
#     rewrite_config,
#     start=0,
#     limit=None,
# )
# rewrite_report

In [ ]:
# import json


# rewritten_path = "../data/datasets/mquake/rewritten"
# mquake_rewritten.save_to_disk(rewritten_path)

# with open(f"{rewritten_path}/failures.json", "w", encoding="utf-8") as file:
#     json.dump(rewrite_failures, file, ensure_ascii=False, indent=2)

# with open(f"{rewritten_path}/report.json", "w", encoding="utf-8") as file:
#     json.dump(rewrite_report, file, ensure_ascii=False, indent=2)

In [ ]:
# import json
# from datasets import load_from_disk


# rewritten_path = "../data/datasets/mquake/rewritten"
# mquake_rewritten = load_from_disk(rewritten_path)
# with open(f"{rewritten_path}/failures.json", encoding="utf-8") as file:
#     rewrite_failures = json.load(file)
# with open(f"{rewritten_path}/report.json", encoding="utf-8") as file:
#     rewrite_report = json.load(file)
# mquake_rewritten

### Final Validation

In [ ]:
# import re
# from collections import Counter
# from copy import deepcopy

# from datasets import Dataset


# def normalize_answer_values(values: list[str]) -> set[str]:
#     output = set()
#     for value in values:
#         marker = prompt_key(value)
#         if marker:
#             output.add(marker)
#     return output


# def error_category(error: str) -> str:
#     error = re.sub(r"^prompt \d+: ", "", error)
#     if error.startswith("missing chain-start subject"):
#         return "missing chain-start subject"
#     if error.startswith("duplicates prompts from source rows"):
#         return "global duplicate prompts"
#     return error


# def validate_record_structure(record: dict) -> list[str]:
#     errors = []
#     portability = record["portability"]
#     true_hops = portability["hops"]["true"]
#     new_hops = portability["hops"]["new"]

#     if len(true_hops) != 2 or len(new_hops) != 2:
#         return ["expected exactly two true and new hops"]
#     for state, hops in (("true", true_hops), ("new", new_hops)):
#         if hops[0]["answers"]["id"] != hops[1]["subject"]["id"]:
#             errors.append(f"{state} hop chain is disconnected")
#         stored_answers = normalize_answer_values(portability["answers"][state])
#         hop_answers = normalize_answer_values(hops[-1]["answers"]["str"])
#         if not stored_answers.intersection(hop_answers):
#             errors.append(f"{state} portability answer does not match the final hop")
#     if [hop["relation"]["id"] for hop in true_hops] != [hop["relation"]["id"] for hop in new_hops]:
#         errors.append("true and new relation chains differ")

#     edit = record["edit"]
#     true_matches = [
#         index
#         for index, hop in enumerate(true_hops)
#         if hop["subject"]["id"] == edit["subject"]["id"]
#         and hop["relation"]["id"] == edit["relation"]["id"]
#         and hop["answers"]["id"] == edit["answer"]["true"]["id"]
#     ]
#     new_matches = [
#         index
#         for index, hop in enumerate(new_hops)
#         if hop["subject"]["id"] == edit["subject"]["id"]
#         and hop["relation"]["id"] == edit["relation"]["id"]
#         and hop["answers"]["id"] == edit["answer"]["new"]["id"]
#     ]
#     if len(true_matches) != 1 or true_matches != new_matches:
#         errors.append("edit does not match exactly one consistent hop position")
#     return errors


# def finalize_dataset(
#     dataset,
#     rewrite_failures: list[dict],
#     start: int = 0,
# ) -> tuple[Dataset, list[dict], dict]:
#     failures_by_index = {failure["index"]: failure for failure in rewrite_failures}
#     final_rows = []
#     final_failures = []
#     seen_prompts = {}
#     recovered_failures = 0
#     recovery_counts = Counter()
#     repair_counts = Counter()
#     error_counts = Counter()

#     for slot, source_row in enumerate(dataset):
#         source_index = start + slot
#         record = prepare_record(source_row)
#         rewrite_failure = failures_by_index.get(source_index)
#         recovered = False

#         if rewrite_failure:
#             rejected = rewrite_failure.get("rewritten") or {}
#             rejected_prompts = rejected.get("prompts") if isinstance(rejected, dict) else None
#             candidate_sets = []
#             if isinstance(rejected_prompts, list):
#                 candidate_sets.append(("rejected_output", rejected_prompts))
#             candidate_sets.append(("original_prompts", record["portability"].get("prompts", [])))
#             candidate_sets.append(("deterministic_fallback", build_fallback_prompts(record)))
#             errors = []
#             recovery_source = None
#             for source_name, prompts in candidate_sets:
#                 candidate_errors = validate_rewritten_prompts(record, prompts)
#                 if not candidate_errors:
#                     record["portability"]["prompts"] = prompts
#                     recovered = True
#                     recovery_source = source_name
#                     break
#             if not recovered:
#                 errors = [*rewrite_failure.get("errors", []), "rewrite did not produce an acceptable prompt set"]
#         else:
#             errors = validate_rewritten_prompts(record, record["portability"].get("prompts", []))

#         if errors:
#             fallback_prompts = build_fallback_prompts(record)
#             fallback_errors = validate_rewritten_prompts(record, fallback_prompts)
#             if not fallback_errors:
#                 record["portability"]["prompts"] = fallback_prompts
#                 errors = []
#                 recovered = True
#                 recovery_source = "final_deterministic_repair"
#                 repair_counts["final_deterministic_repair"] += 1
#             else:
#                 errors = list(dict.fromkeys([*errors, *fallback_errors]))

#         errors.extend(validate_record_structure(record))
#         prompt_keys = [prompt_key(prompt) for prompt in record["portability"].get("prompts", [])]
#         duplicate_owners = sorted({seen_prompts[key] for key in prompt_keys if key in seen_prompts})
#         if duplicate_owners:
#             errors.append(f"duplicates prompts from source rows {duplicate_owners}")

#         if errors:
#             for error in errors:
#                 error_counts[error_category(error)] += 1
#             final_failures.append({
#                 "index": source_index,
#                 "errors": errors,
#                 "record": compact_record(record, include_prompts=True),
#             })
#             continue

#         if recovered:
#             recovered_failures += 1
#             recovery_counts[recovery_source] += 1
#         for key in prompt_keys:
#             seen_prompts[key] = source_index
#         final_rows.append(finalize_record(record))

#     report = {
#         "input_records": len(dataset),
#         "rewrite_failures": len(rewrite_failures),
#         "recovered_failures": recovered_failures,
#         "recovered_by": dict(recovery_counts),
#         "deterministic_repairs": dict(repair_counts),
#         "discarded_records": len(final_failures),
#         "invalid_records": 0,
#         "final_records": len(final_rows),
#         "errors": dict(error_counts),
#     }
#     return Dataset.from_list(final_rows), final_failures, report


# mquake_final, final_failures, final_report = finalize_dataset(
#     mquake_rewritten,
#     rewrite_failures,
#     start=rewrite_report["selected_start"],
# )
# final_report

In [ ]:
# import json
# import os


# final_path = "../data/datasets/mquake/final"
# os.makedirs(final_path, exist_ok=True)

# mquake_final.save_to_disk(final_path)
# with open(f"{final_path}/failures.json", "w", encoding="utf-8") as file:
#     json.dump(final_failures, file, ensure_ascii=False, indent=2)
# with open(f"{final_path}/report.json", "w", encoding="utf-8") as file:
#     json.dump(final_report, file, ensure_ascii=False, indent=2)


In [ ]:
# from datasets import load_from_disk


# mquake = load_from_disk("../data/datasets/mquake/final")
# mquake

## UniEdit

### Raw Snapshot

In [ ]:
# from huggingface_hub import snapshot_download


# snapshot_download(
#     repo_id="qizhou/UniEdit",
#     repo_type="dataset",
#     local_dir="../data/datasets/uniedit/raw",
#     allow_patterns=["*.json"],
# )

In [ ]:
# from glob import glob


# uniedit_raw = {
#     split: sorted(glob(f"../data/datasets/uniedit/raw/{split}/*.json"))
#     for split in ("train", "test")
# }
# {split: len(source_files) for split, source_files in uniedit_raw.items()}

### Merge Repeated Edits

In [ ]:
# import json
# import os
# from tqdm.auto import tqdm
# from typing import Any


# def normalize_uniedit_text(text) -> str:
#     return " ".join(str(text).split()).casefold()


# def extend_unique_uniedit_texts(output: list[str], values: list[Any]) -> None:
#     seen = {normalize_uniedit_text(value) for value in output}
#     for value in values:
#         if value is None or not str(value).strip():
#             continue
#         value = str(value).strip()
#         key = normalize_uniedit_text(value)
#         if key not in seen:
#             seen.add(key)
#             output.append(value)


# def get_uniedit_tail_key(tail: dict[str, Any]) -> tuple[str, Any]:
#     value = tail["value"]
#     if tail["datatype"] == "wikibase-item":
#         value = value["id"]
#     else:
#         value = json.dumps(value, sort_keys=True, ensure_ascii=True)
#     return tail["datatype"], value


# def get_uniedit_pair_key(edit: dict[str, Any]) -> tuple[str, str]:
#     return edit["head_entity"]["id"], edit["property"]["id"]


# def get_uniedit_fact_key(edit: dict[str, Any]) -> tuple[Any, ...]:
#     return (*get_uniedit_pair_key(edit), *get_uniedit_tail_key(edit["tail_entity"]))


# def get_uniedit_tail_names(tail: dict[str, Any]) -> list[str | None]:
#     if tail["datatype"] != "wikibase-item":
#         return []
#     value = tail["value"]
#     return [value.get("label"), *(value.get("aliases") or [])]


# source_files = [
#     source_file
#     for split_files in uniedit_raw.values()
#     for source_file in split_files
# ]
# pair_objects = {}

# for source_file in tqdm(source_files, desc="Find multi-object edits"):
#     with open(source_file, encoding="utf-8") as file:
#         domain_records = json.load(file)
#     for record in domain_records.values():
#         edit = record["edit"]
#         pair_objects.setdefault(get_uniedit_pair_key(edit), set()).add(
#             get_uniedit_tail_key(edit["tail_entity"])
#         )

# multi_object_pairs = {
#     key for key, objects in pair_objects.items() if len(objects) > 1
# }
# uniedit_groups = {}
# excluded_records = 0

# for source_file in tqdm(source_files, desc="Merge UniEdit facts"):
#     with open(source_file, encoding="utf-8") as file:
#         domain_records = json.load(file)
#     for record in domain_records.values():
#         edit = record["edit"]
#         if get_uniedit_pair_key(edit) in multi_object_pairs:
#             excluded_records += 1
#             continue
#         key = get_uniedit_fact_key(edit)
#         if key not in uniedit_groups:
#             merged_edit = {
#                 key: value
#                 for key, value in edit.items()
#                 if key not in ("prompt", "target")
#             }
#             merged_edit["prompts"] = []
#             merged_edit["targets"] = []
#             uniedit_groups[key] = {
#                 "edit": merged_edit,
#                 "generality": [],
#                 "locality": [],
#             }
#         group = uniedit_groups[key]
#         extend_unique_uniedit_texts(group["edit"]["prompts"], [edit["prompt"]])
#         extend_unique_uniedit_texts(
#             group["edit"]["targets"],
#             [edit["target"], *get_uniedit_tail_names(edit["tail_entity"])],
#         )
#         group["generality"].extend(record["generality"].values())
#         group["locality"].extend(record["locality"].values())

# uniedit_merged = list(uniedit_groups.values())
# os.makedirs("../data/datasets/uniedit/merged", exist_ok=True)
# with open("../data/datasets/uniedit/merged/data.jsonl", "w", encoding="utf-8") as file:
#     for record in uniedit_merged:
#         file.write(json.dumps(record, ensure_ascii=False, separators=(",", ":")) + "\n")

# {
#     "multi_object_pairs": len(multi_object_pairs),
#     "excluded_records": excluded_records,
#     "merged_records": len(uniedit_merged),
# }

In [ ]:
# import json


# with open("../data/datasets/uniedit/merged/data.jsonl", encoding="utf-8") as file:
#     uniedit_merged = [json.loads(line) for line in file if line.strip()]
# len(uniedit_merged)

### Evidence Cleaning

In [ ]:
# import json
# import os
# import re
# import unicodedata
# from collections import Counter, defaultdict
# from tqdm.auto import tqdm
# from typing import Any


# def normalize_uniedit_text(text) -> str:
#     return " ".join(str(text).split()).casefold()


# def get_uniedit_prompt_key(text) -> str:
#     text = unicodedata.normalize("NFKC", str(text)).casefold()
#     text = "".join(
#         character
#         if character.isalnum()
#         or unicodedata.category(character).startswith("S")
#         else " "
#         for character in text
#     )
#     return " ".join(text.split())


# def extend_unique_uniedit_texts(output: list[str], values: list[Any]) -> None:
#     seen = {normalize_uniedit_text(value) for value in output}
#     for value in values:
#         if value is None or not str(value).strip():
#             continue
#         value = str(value).strip()
#         key = normalize_uniedit_text(value)
#         if key not in seen:
#             seen.add(key)
#             output.append(value)


# def get_uniedit_answer_targets(target: str, tail: dict[str, Any]) -> list[str]:
#     targets = []
#     tail_names = (
#         [
#             tail["value"].get("label"),
#             *(tail["value"].get("aliases") or []),
#         ]
#         if tail["datatype"] == "wikibase-item"
#         else []
#     )
#     extend_unique_uniedit_texts(targets, [target, *tail_names])
#     return targets


# UNIEDIT_ANSWER_ARTICLES = {"a", "an", "the"}


# def get_uniedit_words(text) -> tuple[str, ...]:
#     text = unicodedata.normalize("NFKC", str(text)).casefold()
#     return tuple(re.findall(r"\w+", text))


# def get_uniedit_answer_keys(text) -> set[str]:
#     text = unicodedata.normalize("NFKC", str(text)).casefold()
#     words = get_uniedit_words(text)
#     keys = {"".join(words)} if words else set()
#     compact = "".join(
#         character
#         for character in text
#         if character.isalnum() or unicodedata.category(character).startswith("S")
#     )
#     if compact:
#         keys.add(compact)
#     if len(words) > 1 and words[0] in UNIEDIT_ANSWER_ARTICLES:
#         keys.add("".join(words[1:]))
#     return keys


# def get_uniedit_entity_texts(entity: dict[str, Any]) -> list[str]:
#     return [
#         value
#         for value in [entity.get("label"), *(entity.get("aliases") or [])]
#         if value is not None and str(value).strip()
#     ]


# def get_uniedit_entity_answers(entity: dict[str, Any]) -> set[str]:
#     answers = set()
#     for value in get_uniedit_entity_texts(entity):
#         answers.update(get_uniedit_answer_keys(value))
#     return answers


# def contains_uniedit_text(text: str, value: str) -> bool:
#     text_words = get_uniedit_words(text)
#     value_words = get_uniedit_words(value)
#     return bool(value_words) and any(
#         text_words[index : index + len(value_words)] == value_words
#         for index in range(len(text_words) - len(value_words) + 1)
#     )


# def has_valid_uniedit_subject(hop: dict[str, Any]) -> bool:
#     return any(
#         contains_uniedit_text(hop["prompt"], value)
#         for value in get_uniedit_entity_texts(hop["head_entity"])
#     )


# def get_used_uniedit_subject_keys(hop: dict[str, Any]) -> set[str]:
#     return {
#         get_uniedit_prompt_key(value)
#         for value in get_uniedit_entity_texts(hop["head_entity"])
#         if contains_uniedit_text(hop["prompt"], value)
#         and get_uniedit_prompt_key(value)
#     }


# def get_uniedit_tail_answers(tail: dict[str, Any]) -> set[str]:
#     value = tail["value"]
#     if tail["datatype"] == "wikibase-item":
#         return get_uniedit_entity_answers(value)
#     if isinstance(value, str):
#         return get_uniedit_answer_keys(value)
#     if isinstance(value, dict) and isinstance(value.get("text"), str):
#         return get_uniedit_answer_keys(value["text"])
#     return set()


# def has_valid_uniedit_target(
#     hop: dict[str, Any],
#     expected_targets: tuple[str, ...] = (),
# ) -> bool:
#     target_answers = get_uniedit_answer_keys(hop["target"])
#     tail_answers = get_uniedit_tail_answers(hop["tail_entity"])
#     if tail_answers:
#         for value in expected_targets:
#             if value is not None and str(value).strip():
#                 tail_answers.update(get_uniedit_answer_keys(value))
#         return bool(target_answers & tail_answers)
#     head_answers = get_uniedit_entity_answers(hop["head_entity"])
#     return not bool(target_answers & head_answers)


# def is_forward_uniedit_path(path: list[str]) -> bool:
#     return path[-2:] in (["f", "s"], ["f", "m"])


# def get_uniedit_path_containers(item: dict[str, Any]) -> list[dict[str, Any]]:
#     if item["path_type"] == "single":
#         return [item]
#     if item["path_type"] == "double":
#         return [item["single_path1"], item["single_path2"]]
#     raise ValueError(f"Unexpected path type: {item['path_type']}")


# def add_uniedit_evidence(
#     evidence: list[dict[str, Any]],
#     prompt: str,
#     targets: list[str],
# ) -> None:
#     key = normalize_uniedit_text(prompt)
#     for candidate in evidence:
#         if normalize_uniedit_text(candidate["prompt"]) == key:
#             extend_unique_uniedit_texts(candidate["targets"], targets)
#             return
#     evidence.append({"prompt": prompt, "targets": list(targets)})


# def get_uniedit_evidence(
#     item: dict[str, Any],
#     edit: dict[str, Any],
#     role: str,
#     edit_pair_keys: set[tuple[str, str]],
#     edit_subject_keys_by_property: dict[str, set[str]],
#     stats: Counter,
# ) -> list[dict[str, Any]]:
#     expected_path = (
#         ["h_e", "r_e", "t_e"]
#         if role == "generality"
#         else ["h_l", "r_l", "t_l"]
#     )
#     edit_fact_key = (
#         edit["head_entity"]["id"],
#         edit["property"]["id"],
#         edit["tail_entity"]["datatype"],
#         edit["tail_entity"]["value"].get("id")
#         if edit["tail_entity"]["datatype"] == "wikibase-item"
#         else json.dumps(edit["tail_entity"]["value"], sort_keys=True, ensure_ascii=True),
#     )
#     edit_head = edit["head_entity"]["id"]
#     edit_property = edit["property"]["id"]
#     evidence = []

#     for container in get_uniedit_path_containers(item):
#         for hop in container.get("one_hops", []):
#             if hop["path"] != expected_path:
#                 continue
#             stats[f"{role}_candidate_hops"] += 1
#             if role == "generality":
#                 hop_fact_key = (
#                     hop["head_entity"]["id"],
#                     hop["property"]["id"],
#                     hop["tail_entity"]["datatype"],
#                     hop["tail_entity"]["value"].get("id")
#                     if hop["tail_entity"]["datatype"] == "wikibase-item"
#                     else json.dumps(hop["tail_entity"]["value"], sort_keys=True, ensure_ascii=True),
#                 )
#                 valid_hop = hop_fact_key == edit_fact_key
#                 if not valid_hop:
#                     stats["generality_fact_mismatch"] += 1
#             else:
#                 hop_pair_key = (
#                     hop["head_entity"]["id"],
#                     hop["property"]["id"],
#                 )
#                 valid_hop = hop["property"]["id"] == edit_property
#                 if not valid_hop:
#                     stats["locality_property_mismatch"] += 1
#                 elif hop["head_entity"]["id"] == edit_head:
#                     valid_hop = False
#                     stats["locality_same_edit_subject"] += 1
#                 elif hop_pair_key in edit_pair_keys:
#                     valid_hop = False
#                     stats["locality_edit_pair_conflict"] += 1
#                 elif get_used_uniedit_subject_keys(hop) & edit_subject_keys_by_property.get(
#                     hop["property"]["id"], set()
#                 ):
#                     valid_hop = False
#                     stats["locality_edit_subject_surface_conflict"] += 1
#             if not valid_hop:
#                 continue
#             expected_targets = edit["targets"] if role == "generality" else ()
#             if not has_valid_uniedit_subject(hop):
#                 stats[f"{role}_subject_grounding_failure"] += 1
#                 continue
#             if not has_valid_uniedit_target(hop, expected_targets):
#                 stats[f"{role}_target_validation_failure"] += 1
#                 continue
#             add_uniedit_evidence(
#                 evidence,
#                 hop["prompt"],
#                 get_uniedit_answer_targets(hop["target"], hop["tail_entity"]),
#             )
#             stats[f"{role}_retained_hops"] += 1

#     if (
#         item["path_type"] == "single"
#         and len(item["paths"]) == 1
#         and len(item["one_hops"]) == 1
#         and item["paths"][0][:3] == expected_path
#         and is_forward_uniedit_path(item["paths"][0])
#         and evidence
#     ):
#         hop_evidence = evidence
#         evidence = []
#         direct_targets = [item["target"]]
#         for candidate in hop_evidence:
#             extend_unique_uniedit_texts(direct_targets, candidate["targets"])
#         add_uniedit_evidence(evidence, item["prompt"], direct_targets)
#         for candidate in hop_evidence:
#             add_uniedit_evidence(
#                 evidence, candidate["prompt"], candidate["targets"]
#             )
#     return evidence


# def clean_uniedit_record(
#     record: dict[str, Any],
#     edit_pair_keys: set[tuple[str, str]],
#     edit_subject_keys_by_property: dict[str, set[str]],
#     stats: Counter,
# ) -> dict[str, Any]:
#     edit = record["edit"]
#     generality = []
#     for item in record["generality"]:
#         for evidence in get_uniedit_evidence(
#             item, edit, "generality", edit_pair_keys,
#             edit_subject_keys_by_property, stats
#         ):
#             add_uniedit_evidence(
#                 generality, evidence["prompt"], evidence["targets"]
#             )
#     locality = []
#     for item in record["locality"]:
#         if item["loc_type"] != "property of edit":
#             continue
#         for evidence in get_uniedit_evidence(
#             item, edit, "locality", edit_pair_keys,
#             edit_subject_keys_by_property, stats
#         ):
#             add_uniedit_evidence(
#                 locality, evidence["prompt"], evidence["targets"]
#             )
#     return {"edit": edit, "generality": generality, "locality": locality}


# uniedit_merged_path = "../data/datasets/uniedit/merged/data.jsonl"
# uniedit_edit_pair_keys = set()
# uniedit_edit_subject_keys_by_property = defaultdict(set)
# uniedit_merged_records = 0
# with open(uniedit_merged_path, encoding="utf-8") as file:
#     for line in tqdm(file, total=289972, desc="Index UniEdit edits"):
#         if not line.strip():
#             continue
#         record = json.loads(line)
#         edit = record["edit"]
#         property_id = edit["property"]["id"]
#         uniedit_edit_pair_keys.add((edit["head_entity"]["id"], property_id))
#         uniedit_edit_subject_keys_by_property[property_id].update(
#             key
#             for value in get_uniedit_entity_texts(edit["head_entity"])
#             if (key := get_uniedit_prompt_key(value))
#         )
#         uniedit_merged_records += 1

# uniedit_clean_stats = Counter()
# uniedit_clean_records = 0
# records_with_generality = 0
# records_with_locality = 0
# generality_evidence = 0
# locality_evidence = 0
# os.makedirs("../data/datasets/uniedit/clean", exist_ok=True)
# with open(uniedit_merged_path, encoding="utf-8") as source_file, open(
#     "../data/datasets/uniedit/clean/data.jsonl", "w", encoding="utf-8"
# ) as output_file:
#     for line in tqdm(
#         source_file,
#         total=uniedit_merged_records,
#         desc="Clean UniEdit facts",
#     ):
#         if not line.strip():
#             continue
#         clean_record = clean_uniedit_record(
#             json.loads(line),
#             uniedit_edit_pair_keys,
#             uniedit_edit_subject_keys_by_property,
#             uniedit_clean_stats,
#         )
#         output_file.write(
#             json.dumps(clean_record, ensure_ascii=False, separators=(",", ":"))
#             + "\n"
#         )
#         uniedit_clean_records += 1
#         records_with_generality += bool(clean_record["generality"])
#         records_with_locality += bool(clean_record["locality"])
#         generality_evidence += len(clean_record["generality"])
#         locality_evidence += len(clean_record["locality"])

# uniedit_clean_report = {
#     "input_records": uniedit_merged_records,
#     "clean_records": uniedit_clean_records,
#     "records_with_generality": records_with_generality,
#     "records_with_locality": records_with_locality,
#     "generality_evidence": generality_evidence,
#     "locality_evidence": locality_evidence,
#     "validation": dict(uniedit_clean_stats),
# }
# with open("../data/datasets/uniedit/clean/report.json", "w", encoding="utf-8") as file:
#     json.dump(uniedit_clean_report, file, ensure_ascii=False, indent=2)
# uniedit_clean_report


In [ ]:
# import json


# with open("../data/datasets/uniedit/clean/data.jsonl", encoding="utf-8") as file:
#     uniedit_clean = [json.loads(line) for line in file if line.strip()]
# len(uniedit_clean)

### Final Dataset

In [ ]:
# import json
# import os
# import unicodedata
# from collections import defaultdict
# from datasets import Dataset
# from typing import Any


# def normalize_uniedit_text(text) -> str:
#     return " ".join(str(text).split()).casefold()


# def get_uniedit_prompt_key(text) -> str:
#     text = unicodedata.normalize("NFKC", str(text)).casefold()
#     text = "".join(
#         character
#         if character.isalnum()
#         or unicodedata.category(character).startswith("S")
#         else " "
#         for character in text
#     )
#     return " ".join(text.split())


# def extend_unique_uniedit_texts(output: list[str], values: list[Any]) -> None:
#     seen = {normalize_uniedit_text(value) for value in output}
#     for value in values:
#         if value is None or not str(value).strip():
#             continue
#         value = str(value).strip()
#         key = normalize_uniedit_text(value)
#         if key not in seen:
#             seen.add(key)
#             output.append(value)


# def merge_uniedit_final_rows(left: dict[str, Any], right: dict[str, Any]) -> None:
#     extend_unique_uniedit_texts(left["edit"]["targets"], right["edit"]["targets"])
#     seen_prompts = {get_uniedit_prompt_key(left["edit"]["prompt"])}
#     generality = []
#     for prompt in [*left["generality"], *right["generality"]]:
#         key = get_uniedit_prompt_key(prompt)
#         if key not in seen_prompts:
#             seen_prompts.add(key)
#             generality.append(prompt)
#     locality = []
#     for item in [*left["locality"], *right["locality"]]:
#         key = get_uniedit_prompt_key(item["prompt"])
#         if key in seen_prompts:
#             for candidate in locality:
#                 if get_uniedit_prompt_key(candidate["prompt"]) == key:
#                     extend_unique_uniedit_texts(candidate["targets"], item["targets"])
#                     break
#             continue
#         seen_prompts.add(key)
#         locality.append({"prompt": item["prompt"], "targets": list(item["targets"])})
#     left["generality"] = generality
#     left["locality"] = locality


# def build_uniedit_candidate(record: dict[str, Any]) -> dict[str, Any] | None:
#     edit_prompt = record["edit"]["prompts"][0]
#     edit_targets = list(record["edit"]["targets"])
#     for evidence in record["generality"]:
#         extend_unique_uniedit_texts(edit_targets, evidence["targets"])

#     seen_prompts = {get_uniedit_prompt_key(edit_prompt)}
#     generality = []
#     for prompt in [
#         *record["edit"]["prompts"][1:],
#         *(item["prompt"] for item in record["generality"]),
#     ]:
#         key = get_uniedit_prompt_key(prompt)
#         if key and key not in seen_prompts:
#             seen_prompts.add(key)
#             generality.append(prompt)

#     locality = []
#     for item in record["locality"]:
#         key = get_uniedit_prompt_key(item["prompt"])
#         if not key or key in seen_prompts:
#             continue
#         seen_prompts.add(key)
#         locality.append({
#             "prompt": item["prompt"],
#             "targets": list(item["targets"]),
#         })

#     if not generality or not locality or not edit_targets:
#         return None
#     return {
#         "edit": {"prompt": edit_prompt, "targets": edit_targets},
#         "generality": generality,
#         "locality": locality,
#     }


# uniedit_candidates = []
# uniedit_clean_records = 0
# with open("../data/datasets/uniedit/clean/data.jsonl", encoding="utf-8") as file:
#     for line in file:
#         if not line.strip():
#             continue
#         uniedit_clean_records += 1
#         candidate = build_uniedit_candidate(json.loads(line))
#         if candidate is not None:
#             uniedit_candidates.append(candidate)

# rows_by_edit_prompt = {}
# ambiguous_edit_prompts = set()
# merged_edit_prompt_rows = 0
# for row in uniedit_candidates:
#     key = get_uniedit_prompt_key(row["edit"]["prompt"])
#     if key not in rows_by_edit_prompt:
#         rows_by_edit_prompt[key] = row
#         continue
#     existing_targets = {
#         normalize_uniedit_text(value)
#         for value in rows_by_edit_prompt[key]["edit"]["targets"]
#     }
#     new_targets = {
#         normalize_uniedit_text(value)
#         for value in row["edit"]["targets"]
#     }
#     if existing_targets & new_targets:
#         merge_uniedit_final_rows(rows_by_edit_prompt[key], row)
#         merged_edit_prompt_rows += 1
#     else:
#         ambiguous_edit_prompts.add(key)

# uniedit_rows = [
#     row
#     for key, row in rows_by_edit_prompt.items()
#     if key not in ambiguous_edit_prompts
# ]

# edit_prompt_owners = {
#     get_uniedit_prompt_key(row["edit"]["prompt"]): index
#     for index, row in enumerate(uniedit_rows)
# }
# generality_owners = defaultdict(set)
# for index, row in enumerate(uniedit_rows):
#     for prompt in row["generality"]:
#         generality_owners[get_uniedit_prompt_key(prompt)].add(index)

# ambiguous_generality_keys = {
#     key for key, owners in generality_owners.items() if len(owners) > 1
# }
# generality_edit_collision_keys = set(generality_owners) & set(edit_prompt_owners)
# invalid_generality_keys = ambiguous_generality_keys | generality_edit_collision_keys
# removed_generality_prompts = 0
# for row in uniedit_rows:
#     retained = []
#     for prompt in row["generality"]:
#         if get_uniedit_prompt_key(prompt) in invalid_generality_keys:
#             removed_generality_prompts += 1
#             continue
#         retained.append(prompt)
#     row["generality"] = retained

# positive_prompt_keys = set(edit_prompt_owners)
# positive_prompt_keys.update(
#     get_uniedit_prompt_key(prompt)
#     for row in uniedit_rows
#     for prompt in row["generality"]
# )
# removed_locality_prompts = 0
# for row in uniedit_rows:
#     retained = []
#     for item in row["locality"]:
#         if get_uniedit_prompt_key(item["prompt"]) in positive_prompt_keys:
#             removed_locality_prompts += 1
#             continue
#         retained.append(item)
#     row["locality"] = retained

# records_without_generality = sum(not row["generality"] for row in uniedit_rows)
# records_without_locality = sum(not row["locality"] for row in uniedit_rows)
# uniedit_rows = [
#     row for row in uniedit_rows if row["generality"] and row["locality"]
# ]

# final_positive_owners = defaultdict(set)
# final_locality_keys = set()
# for index, row in enumerate(uniedit_rows):
#     prompts = [row["edit"]["prompt"], *row["generality"]]
#     all_prompts = [*prompts, *(item["prompt"] for item in row["locality"])]
#     if len({get_uniedit_prompt_key(prompt) for prompt in all_prompts}) != len(all_prompts):
#         raise ValueError(f"Duplicate prompt within final record {index}.")
#     if not row["edit"]["targets"] or any(not item["targets"] for item in row["locality"]):
#         raise ValueError(f"Empty target list within final record {index}.")
#     for prompt in prompts:
#         final_positive_owners[get_uniedit_prompt_key(prompt)].add(index)
#     for item in row["locality"]:
#         final_locality_keys.add(get_uniedit_prompt_key(item["prompt"]))

# ambiguous_final_positive_keys = {
#     key for key, owners in final_positive_owners.items() if len(owners) != 1
# }
# positive_locality_overlap = set(final_positive_owners) & final_locality_keys
# if ambiguous_final_positive_keys or positive_locality_overlap:
#     raise ValueError(
#         "Final UniEdit global prompt validation failed: "
#         f"ambiguous_positive={len(ambiguous_final_positive_keys)} "
#         f"positive_locality_overlap={len(positive_locality_overlap)}"
#     )

# uniedit_final_report = {
#     "input_clean_records": uniedit_clean_records,
#     "candidate_records": len(uniedit_candidates),
#     "merged_same_answer_edit_prompt_rows": merged_edit_prompt_rows,
#     "ambiguous_edit_prompt_keys": len(ambiguous_edit_prompts),
#     "records_after_edit_prompt_resolution": len(edit_prompt_owners),
#     "ambiguous_generality_keys": len(ambiguous_generality_keys),
#     "generality_edit_collision_keys": len(generality_edit_collision_keys),
#     "removed_generality_prompts": removed_generality_prompts,
#     "removed_positive_locality_prompts": removed_locality_prompts,
#     "records_without_generality": records_without_generality,
#     "records_without_locality": records_without_locality,
#     "final_records": len(uniedit_rows),
#     "final_generality_prompts": sum(len(row["generality"]) for row in uniedit_rows),
#     "final_locality_prompts": sum(len(row["locality"]) for row in uniedit_rows),
#     "unique_positive_prompts": len(final_positive_owners),
#     "unique_locality_prompts": len(final_locality_keys),
#     "validation": {
#         "unique_edit_prompts": len({
#             get_uniedit_prompt_key(row["edit"]["prompt"])
#             for row in uniedit_rows
#         }) == len(uniedit_rows),
#         "one_owner_per_positive_prompt": not ambiguous_final_positive_keys,
#         "unique_prompts_within_records": True,
#         "no_positive_locality_overlap": not positive_locality_overlap,
#         "nonempty_generality_and_locality": all(
#             row["generality"] and row["locality"] for row in uniedit_rows
#         ),
#     },
# }

# uniedit_final = Dataset.from_list(uniedit_rows)
# uniedit_final_path = "../data/datasets/uniedit/final"
# uniedit_final.save_to_disk(uniedit_final_path)
# with open(f"{uniedit_final_path}/report.json", "w", encoding="utf-8") as file:
#     json.dump(uniedit_final_report, file, ensure_ascii=False, indent=2)
# uniedit_final_report


In [ ]:
# from datasets import load_from_disk


# uniedit = load_from_disk("../data/datasets/uniedit/final")
# uniedit

## Publishing

### Edit Success - Locality

In [ ]:
# from datasets import load_from_disk


# es_loc = load_from_disk("../data/datasets/counterfact/final")
# es_loc

In [ ]:
# es_loc.push_to_hub(
#     "Savassan/KnowledgeSurgeon",
#     config_name="es_loc",
#     split="data",
#     private=True,
# )

### Portability

In [ ]:
# from datasets import load_from_disk


# port = load_from_disk("../data/datasets/mquake/final")
# port

In [ ]:
# port.push_to_hub(
#     "Savassan/KnowledgeSurgeon",
#     config_name="port",
#     split="data",
#     private=True,
# )

### Similarity

In [ ]:
# from datasets import load_from_disk


# sim = load_from_disk("../data/datasets/uniedit/final")
# sim

In [ ]:
# sim.push_to_hub(
#     "Savassan/KnowledgeSurgeon",
#     config_name="similarity",
#     split="data",
#     private=True,
# )